<a href="https://colab.research.google.com/github/jesusvasquez-tec/analitica-empresarial-integrada/blob/main/Laboratorio%20Calificado%206/LC6_AEI_Grupo3_2026_02.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ANALÍTICA EMPRESARIAL INTEGRADA
## LABORATORIO CALIFICADO N.° 06 — Optimización de inventario y monitoreo de anomalías

**Semanas 11 y 12 · TECSUP 2026-II**
**Docente:** PILAR ROCÍO SAYÁN MEJÍA
**Dataset real:** Online Retail II — UCI Machine Learning Repository (líneas de factura de un mayorista; clientes fuera del Reino Unido)
**Objetivo:** decidir cómo repartir un inventario limitado entre 12 productos y qué semanas de ventas investigar por registros inusuales, con optimización entera, validación temporal sin información futura y una recomendación integrada.

> Esta es una evaluación **grupal** de 16 puntos (más 4 individuales). La **introducción** de este notebook — instalación, parámetros, obtención de los datos y trazabilidad — ya está programada. A partir del Ejercicio 1, **el equipo escribe su propio código** y responde cada pregunta.

## Trabajo grupal y entrega

Ejecute el notebook en **Google Colab** de inicio a fin. Las respuestas, el informe y la recomendación quedan en este mismo archivo.

**Al final de la sesión:**
1. Suba este notebook, ejecutado y con todas las respuestas, a la carpeta `lc6/` del repositorio del equipo en GitHub.
2. Conserve la versión en el Colab del equipo y compártala con permiso de lectura.
3. Entregue **ambos URLs** en la actividad de la semana.

**Identificación del equipo**
- Equipo N.°: Grupo3
- Sección: 6C28A
- Fecha: 8 noviembre 2026
- URL de GitHub:
- URL de Colab: https://drive.google.com/file/d/1V0OnDH4Zthgpua-PtsjeJgfYOwCFcOGb/view?usp=sharing
- Uso de IA declarado (herramienta y partes del trabajo):

# **Declaración de uso de asistentes de IA**

Este laboratorio fue desarrollado con apoyo de **Qwen (asistente de IA)** en las siguientes partes:

- **Revisión y ajuste de la lógica de código** en los ejercicios
- **Redacción de las interpretaciones** en markdown después de cada bloque de código, a partir de los outputs obtenidos.
- **Construcción del informe y preguntas** (secciones A a C), integrando los resultados del análisis y recomendaciones para la empresa

**Todo el código fue revisado, ejecutado y comprendido por el equipo**, y la documentación fue redactada con palabras propias justificando las decisiones metodológicas.

| **N.°** | **Apellidos y nombres**  | **Código** | **Aporte principal en este laboratorio**                                                                                                                                                                                                                                                                      |
| ------: | ------------------------ | ---------: | ------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
|       1 | Alvino Carhuaricra, Enzo |     118430 | **Código:** Ejercicio 1 (Trazabilidad, calidad y alcance) y Ejercicio 2 (Modelo de optimización de inventario).<br><br>**Análisis:** Redacción de las preguntas 1.1 a 2.4.<br><br>**Preguntas:** Responder la sección A |
|       2 | Cordova Agurto, Isabel   |     118082 | **Código:** Ejercicio 3 (Regla intuitiva, valor realizado y sensibilidad) y Ejercicio 4 (Detección de anomalías con línea base ).<br><br>**Análisis:** Redacción de las preguntas 3.1 a 4.4.<br><br>**Preguntas:** Responder la sección B                               |
|       3 | Palomino Acuña, Luis     |     119055 | **Código:** Ejercicio 5 (Carta p semanal y cola de revisión) y Ejercicio 6 ( Decisión integrada y sensibilidad ).<br><br>**Análisis:** Redacción de las preguntas 5.1 a 6.3.<br><br>**Preguntas:** Responder la sección C |
|       4 | Vasquez Espinoza, Jesus  |     110654 | **Código:** Revisión y estandarización de la documentación línea por línea de todo el notebook.<br><br>**Análisis:** Redacción de las conclusiones generales y la Declaración de uso de IA.<br><br>**Informe:** Redacción del informe |


## Normas y buenas prácticas

- Conserve la **trazabilidad** y distinga datos observados, supuestos académicos y resultados simulados.
- **No cambie** los parámetros, los umbrales ni las reglas de decisión después de conocer la evaluación final para favorecer una conclusión.
- Se permite usar asistentes de IA **declarándolo** en la identificación del equipo. Cada línea de código propio debe explicarse con palabras del equipo: qué decisión resuelve, por qué se usa y qué cambiaría si se omitiera.
- Puede reutilizar el trabajo de los laboratorios dirigidos 11 y 12, declarando su origen y adaptándolo a estos datos.
- La docente puede pedir a cualquier integrante que sustente el código y los resultados. La documentación vale 3 de los 16 puntos grupales.
- **Duración:** 190 minutos (15 de receso). Distribución sugerida: lectura y datos, 15; calidad, 20; modelo de optimización, 40; comparación y sensibilidad, 30; receso, 15; anomalías y carta p, 40; decisión integrada, 20; cierre, 10.

# 1. Pregunta de negocio

**La gerencia de operaciones de un mayorista de artículos para el hogar prepara el siguiente periodo para sus clientes de Europa continental: ¿cómo debe repartir su capacidad y su presupuesto entre los 12 productos principales y qué semanas de ventas conviene investigar por registros inusuales?**

El equipo debe recomendar, para **cada decisión**, una acción y sus condiciones, combinando el valor de asignar bien el inventario con la capacidad real del equipo de revisión.

### Diseño del laboratorio
| Elemento | Definición |
|---|---|
| Base de datos | Online Retail II (UCI, ID 502): una tabla de líneas de factura |
| Periodo | Semanas completas de lunes a domingo: 7 de diciembre de 2009 al 4 de diciembre de 2011 (102 semanas) |
| Línea base | Hasta el 3 de julio de 2011 (80 semanas) |
| Evaluación | Del 4 de julio al 4 de diciembre de 2011 (22 semanas) |
| Parte A — unidad | Producto (`StockCode`), demanda semanal estimada con 26 semanas de la línea base |
| Parte B — unidad | Línea de factura; monitoreo semanal de la proporción de alertas |

### Parámetros académicos
Los valores económicos y de capacidad **no provienen del archivo**: son supuestos de evaluación. Están definidos en la celda de parámetros y **no deben modificarse**.

In [ ]:
# Instala en Colab las librerías que no vienen por defecto; la versión de PuLP queda fija para repetir resultados
%pip install -q "ucimlrepo>=0.0.7" "pulp==3.3.0" plotly openpyxl scikit-learn

In [ ]:
import hashlib
import sys
import warnings
from datetime import datetime, timezone
from io import BytesIO
from pathlib import Path
from urllib.request import urlopen
from zipfile import ZipFile

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import pulp as pl
import sklearn
from sklearn.ensemble import IsolationForest
from ucimlrepo import fetch_ucirepo

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
VERSIONES = {"python": sys.version.split()[0], "pandas": pd.__version__, "numpy": np.__version__,
             "pulp": pl.__version__, "scikit-learn": sklearn.__version__}
print(VERSIONES)

{'python': '3.13.18', 'pandas': '2.2.3', 'numpy': '2.4.6', 'pulp': '3.3.0', 'scikit-learn': '1.6.1'}


In [ ]:
# ---------------- PARÁMETROS ACADÉMICOS DEL CASO (no modificar) ----------------
SEMILLA = 2026
np.random.seed(SEMILLA)

# Periodo y línea de tiempo (semanas completas de lunes a domingo)
INICIO = pd.Timestamp("2009-12-07")
FIN = pd.Timestamp("2011-12-05")            # exclusivo: el último día incluido es el 4 de diciembre de 2011
CORTE = pd.Timestamp("2011-07-04")          # línea base: antes de esta fecha; evaluación: desde esta fecha
VENTANA_DEMANDA_INICIO = pd.Timestamp("2011-01-03")   # 26 semanas de la línea base para estimar la demanda
SEMANAS_VENTANA = 26

# Parte A — optimización de inventario
N_PRODUCTOS = 12
FACTOR_COSTO = 0.60               # costo unitario = 60 % del precio mediano (supuesto académico)
CAPACIDAD_PROP = 0.45             # de la demanda semanal total
PRESUPUESTO_PROP = 0.60           # del costo de cubrir toda la demanda
SERVICIO_MINIMO = 0.25            # de la demanda de cada producto (al menos 1 unidad)
CONCENTRACION_MAX = 0.20          # de la capacidad
CAPACIDADES = [0.40, 0.45, 0.55]  # sensibilidad
PRESUPUESTOS = [0.50, 0.60, 0.70]

# Parte B — anomalías, carta p y revisión
N_ARBOLES, MAX_MUESTRAS = 150, 5000
PERCENTIL = 0.99                  # sensibilidad: 0.98 y 0.995
PERCENTILES = [0.98, 0.99, 0.995]
SIGMAS = 3
CAPACIDAD_REVISION = 20           # líneas por semana (supuesto académico)
print("Parámetros cargados.")

Parámetros cargados.


# 2. Obtención de los datos y trazabilidad

Online Retail II se distribuye como un libro de Excel con dos hojas (2009-2010 y 2010-2011). La celda siguiente intenta, en este orden:

1. La API `ucimlrepo` (ID 502).
2. El ZIP oficial de UCI.
3. El archivo `online_retail_II.xlsx` en la carpeta de la sesión, si el equipo lo descargó y lo subió.

Si ninguna funciona, solicite una copia a la docente. **No sustituya la fuente ni genere datos ficticios.** El archivo supera el millón de filas: la carga puede tardar varios minutos.

In [ ]:
try:
    retail = fetch_ucirepo(id=502)
    df = retail.data.original.copy()
    fuente = "API ucimlrepo, UCI ID 502"
except Exception as exc:
    print("API no disponible:", type(exc).__name__)
    try:
        url = "https://archive.ics.uci.edu/static/public/502/online+retail+ii.zip"
        with urlopen(url, timeout=180) as response:
            with ZipFile(BytesIO(response.read())) as z:
                nombre = next(n for n in z.namelist() if n.lower().endswith(".xlsx"))
                df = pd.concat(pd.read_excel(BytesIO(z.read(nombre)), sheet_name=None).values(), ignore_index=True)
        fuente = "ZIP oficial UCI, ID 502"
    except Exception:
        local = Path("online_retail_II.xlsx")
        if not local.exists():
            raise FileNotFoundError("Descargue online_retail_II.xlsx desde UCI y súbalo a la carpeta de Colab, o solicite una copia a la docente.")
        df = pd.concat(pd.read_excel(local, sheet_name=None).values(), ignore_index=True)
        fuente = "Archivo local descargado de UCI"
print("Fuente:", fuente)
print("Dimensión:", df.shape)

API no disponible: DatasetNotFoundError
Fuente: ZIP oficial UCI, ID 502
Dimensión: (1067371, 8)


In [ ]:
# Nombres de columna únicos: el archivo puede traer Invoice o InvoiceNo, Price o UnitPrice, Customer ID o CustomerID
RENOMBRAR = {"Invoice": "InvoiceNo", "Price": "UnitPrice", "Customer ID": "CustomerID"}
df = df.rename(columns={k: v for k, v in RENOMBRAR.items() if k in df.columns})
OBLIGATORIAS = ["InvoiceNo", "StockCode", "Description", "Quantity", "InvoiceDate", "UnitPrice", "Country"]
faltan = [c for c in OBLIGATORIAS if c not in df.columns]
assert not faltan, f"Faltan columnas obligatorias: {faltan}"
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"], errors="coerce")
df["Quantity"] = pd.to_numeric(df["Quantity"], errors="coerce")
df["UnitPrice"] = pd.to_numeric(df["UnitPrice"], errors="coerce")

# Trazabilidad: fuente, momento de la lectura, dimensiones, memoria y huella de las filas
TRAZA = {
    "fuente": fuente,
    "lectura_utc": datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S"),
    "filas": len(df),
    "columnas": df.shape[1],
    "memoria_MB": round(float(df.memory_usage(deep=True).sum()) / 1e6, 1),
    "huella_filas": int(pd.util.hash_pandas_object(df[OBLIGATORIAS].astype(str), index=False).sum() % (10 ** 12)),
    "fecha_min": str(df["InvoiceDate"].min()),
    "fecha_max": str(df["InvoiceDate"].max()),
    "versiones": VERSIONES,
}
for k, v in TRAZA.items():
    print(f"{k}: {v}")
display(df.head(5))

fuente: ZIP oficial UCI, ID 502
lectura_utc: 2026-11-06 17:22:11
filas: 1067371
columnas: 8
memoria_MB: 261.2
huella_filas: 194587644269
fecha_min: 2009-12-01 07:45:00
fecha_max: 2011-12-09 12:50:00
versiones: {'python': '3.13.18', 'pandas': '2.2.3', 'numpy': '2.4.6', 'pulp': '3.3.0', 'scikit-learn': '1.6.1'}


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.9500,"13,085.0000",United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.7500,"13,085.0000",United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.7500,"13,085.0000",United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.1000,"13,085.0000",United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.2500,"13,085.0000",United Kingdom


> **Desde aquí el código es del equipo.** Cada ejercicio tiene sus preguntas, una celda de código vacía para resolverlas y un espacio de **Respuesta** (máximo cinco líneas con cifras propias). Cada resultado se valora una sola vez dentro del criterio correspondiente de la rúbrica. El DataFrame `df` contiene los registros originales: no lo modifique; trabaje sobre copias y documente cada exclusión.

# Ejercicio 1 — Trazabilidad, calidad y alcance

### Pregunta 1.1

Registre la fuente, la fecha de descarga, las dimensiones, el rango de InvoiceDate y el número de países. Presente un cuadro de exclusiones consecutivas con las filas que elimina cada regla: fuera del periodo del 7 de diciembre de 2009 al 4 de diciembre de 2011, facturas de cancelación (prefijo C), cantidad o precio no positivos, código de producto que no empieza con cinco dígitos y país Reino Unido. ¿Cuántas líneas, facturas, productos y semanas completas quedan?

In [ ]:
# EJERCICIO 1.1: TRAZABILIDAD, EXCLUSIONES Y ALCANCE

# Conservamos una copia del DataFrame original para realizar la limpieza.
df_clean = df.copy()

# Registramos las dimensiones y características de la fuente original.
filas_iniciales = len(df_clean)
columnas_iniciales = len(df_clean.columns)
paises_originales = df_clean['Country'].nunique()
fecha_minima = df_clean['InvoiceDate'].min()
fecha_maxima = df_clean['InvoiceDate'].max()

# Recuperamos la información de trazabilidad registrada anteriormente.
fecha_lectura = TRAZA.get('lectura_utc', 'No disponible')

# Definimos el periodo de análisis.
# El límite final es exclusivo para incluir todo el domingo 04/12/2011.
fecha_inicio = pd.Timestamp('2009-12-07')
fecha_fin_exclusiva = pd.Timestamp('2011-12-05')

# REGLA 1: Excluir registros fuera del periodo establecido.
mask_periodo = (
    (df_clean['InvoiceDate'] >= fecha_inicio) &
    (df_clean['InvoiceDate'] < fecha_fin_exclusiva)
)
filas_post_1 = len(df_clean[mask_periodo])
eliminadas_1 = filas_iniciales - filas_post_1
df_clean = df_clean[mask_periodo].copy()

# REGLA 2: Excluir facturas de cancelación identificadas con el prefijo C.
mask_cancel = ~df_clean['InvoiceNo'].astype(str).str.startswith('C')
filas_post_2 = len(df_clean[mask_cancel])
eliminadas_2 = filas_post_1 - filas_post_2
df_clean = df_clean[mask_cancel].copy()

# REGLA 3: Excluir líneas con cantidad o precio unitario no positivos.
mask_valores = (
    (df_clean['Quantity'] > 0) &
    (df_clean['UnitPrice'] > 0)
)
filas_post_3 = len(df_clean[mask_valores])
eliminadas_3 = filas_post_2 - filas_post_3
df_clean = df_clean[mask_valores].copy()

# Limpieza preventiva: eliminamos el '.0' si Pandas leyó algún StockCode numérico como float
df_clean['StockCode'] = df_clean['StockCode'].astype(str).str.replace(r'\.0$', '', regex=True)

# REGLA 4: Excluir códigos de producto que no comienzan con cinco dígitos.
mask_stock = df_clean['StockCode'].astype(str).str.contains(r'^\d{5}', regex=True, na=False)
filas_post_4 = len(df_clean[mask_stock])
eliminadas_4 = filas_post_3 - filas_post_4
df_clean = df_clean[mask_stock].copy()

# REGLA 5: Excluir registros correspondientes al Reino Unido.
mask_pais = df_clean['Country'] != 'United Kingdom'
filas_post_5 = len(df_clean[mask_pais])
eliminadas_5 = filas_post_4 - filas_post_5
df_clean = df_clean[mask_pais].copy()

# Construimos el cuadro de exclusiones consecutivas.
cuadro_exclusiones = pd.DataFrame({
    'Regla de limpieza': [
        '1. Fuera del periodo (07/12/2009 - 04/12/2011)',
        '2. Facturas de cancelación (prefijo C)',
        '3. Cantidad o precio unitario <= 0',
        '4. StockCode no inicia con cinco dígitos',
        '5. País es United Kingdom'
    ],
    'Filas eliminadas': [
        eliminadas_1, eliminadas_2, eliminadas_3, eliminadas_4, eliminadas_5
    ],
    'Filas restantes': [
        filas_post_1, filas_post_2, filas_post_3, filas_post_4, filas_post_5
    ]
})

# Calculamos las métricas finales del conjunto depurado.
lineas_finales = len(df_clean)
facturas_finales = df_clean['InvoiceNo'].nunique()
productos_finales = df_clean['StockCode'].nunique()

# Verificamos la cantidad de semanas completas del intervalo calendario.
dias_periodo = (fecha_fin_exclusiva - fecha_inicio).days
assert dias_periodo % 7 == 0, "El periodo no contiene semanas completas."
semanas_periodo = dias_periodo // 7

# Contamos las semanas distintas con registros después de la limpieza.
semanas_con_registros = (
    df_clean['InvoiceDate']
    .dt.to_period('W-SUN')
    .nunique()
)

# Mostramos la información de trazabilidad de la fuente original.
print("="*34)
print("TRAZABILIDAD DE LA FUENTE ORIGINAL")
print("="*34)
print(f"🔹 Fuente: {fuente}")
print(f"🔹 Fecha de descarga/lectura UTC: {fecha_lectura}")
print(f"🔹 Filas originales: {filas_iniciales:,}")
print(f"🔹 Columnas originales: {columnas_iniciales}")
print(f"🔹 Países distintos originales: {paises_originales}")
print(f"🔹 Rango de InvoiceDate: {fecha_minima} a {fecha_maxima}")

# Mostramos el cuadro de exclusiones consecutivas.
print("\n")
print("="*34)
print("CUADRO DE EXCLUSIONES CONSECUTIVAS")
print("="*34)
display(cuadro_exclusiones)

# Mostramos el resumen final del alcance.
print("\n")
print("="*24)
print("RESUMEN FINAL DEL ALCANCE")
print("="*24)
print(f"🔹 Líneas restantes: {lineas_finales:,}")
print(f"🔹 Facturas únicas restantes: {facturas_finales:,}")
print(f"🔹 Productos únicos (StockCode): {productos_finales}")
print(f"🔹 Semanas calendario del periodo: {semanas_periodo}")
print(f"🔹 Semanas con registros comerciales: {semanas_con_registros}")

TRAZABILIDAD DE LA FUENTE ORIGINAL
🔹 Fuente: ZIP oficial UCI, ID 502
🔹 Fecha de descarga/lectura UTC: 2026-11-06 17:22:11
🔹 Filas originales: 1,067,371
🔹 Columnas originales: 8
🔹 Países distintos originales: 43
🔹 Rango de InvoiceDate: 2009-12-01 07:45:00 a 2011-12-09 12:50:00


CUADRO DE EXCLUSIONES CONSECUTIVAS


,Regla de limpieza,Filas eliminadas,Filas restantes
0,1. Fuera del periodo (07/12/2009 - 04/12/2011),32112,1035259
1,2. Facturas de cancelación (prefijo C),19033,1016226
2,3. Cantidad o precio unitario <= 0,6138,1010088
3,4. StockCode no inicia con cinco dígitos,4700,1005388
4,5. País es United Kingdom,925987,79401




RESUMEN FINAL DEL ALCANCE
🔹 Líneas restantes: 79,401
🔹 Facturas únicas restantes: 3,261
🔹 Productos únicos (StockCode): 3650
🔹 Semanas calendario del periodo: 104
🔹 Semanas con registros comerciales: 102


**Respuesta:**

1.1 **Trazabilidad y alcance**

La fuente es el ZIP oficial UCI (ID 502), leído el 05/11/2026 a las 23:48:52 UTC, con 1,067,371 filas, 8 columnas, 43 países y un rango de fechas del 01/12/2009 al 09/12/2011. Tras aplicar las cinco reglas de exclusión consecutivas (detalladas en el cuadro superior), el alcance final del análisis queda conformado por 79,401 líneas, 3,261 facturas únicas, 3,650 productos distintos y 102 semanas con registros comerciales dentro del periodo.

1.2 **Justificación de exclusiones**

Las facturas con prefijo 'C' y las líneas con cantidad o precio ≤ 0 representan devoluciones o ajustes contables, no ventas netas, por lo que incluirlos distorsionaría la estimación de la demanda real. Códigos como 'POST' (gastos de envío), 'BANK CHARGES' (comisiones) o 'M' (ajustes manuales) se excluyen porque son conceptos de servicio o financieros, no productos físicos sujetos a gestión de inventario.

1.3 **Periodo y limitaciones de los datos**

El periodo establecido (lunes 07/12/2009 a domingo 04/12/2011) abarca exactamente 728 días, es decir, 104 semanas completas de lunes a domingo, lo que evita distorsiones por semanas cortadas al inicio o final del análisis. Con esta base de datos no se puede sostener que el comportamiento de demanda sea extrapolable a otros mercados o al año 2026, y por otro lado que existan quiebres de stock reales, ya que el archivo solo registra líneas de facturación, no niveles de inventario físico.

### Pregunta 1.2

¿Qué representan las facturas de cancelación y las líneas con cantidad o precio no positivos, y por qué no deben contarse como ventas al estimar la demanda de un producto? Muestre tres ejemplos de códigos que no empiezan con cinco dígitos y explique por qué se excluyen.

In [ ]:
# EJERCICIO 1.2: IDENTIFICACIÓN DE CANCELACIONES, VALORES NO POSITIVOS Y CÓDIGOS

# 1. Extraemos ejemplos de facturas de cancelación (prefijo C).
# Se muestran registros del DataFrame original para observar sus características.
ej_cancel = df[
    df['InvoiceNo'].astype(str).str.startswith('C')
][['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'UnitPrice']].head(3)

# 2. Extraemos ejemplos de líneas con cantidad o precio unitario no positivos.
# Se consideran registros donde Quantity <= 0 o UnitPrice <= 0.
ej_no_positivos = df[
    (df['Quantity'] <= 0) | (df['UnitPrice'] <= 0)
][['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'UnitPrice']].head(3)

# 3. Extraemos tres códigos de producto distintos que no comienzan con cinco dígitos.
# Se usa na=False para evitar que valores nulos (NaN) se conviertan en texto 'nan' y sean seleccionados.
mask_no_5digitos = ~df['StockCode'].astype(str).str.contains(r'^\d{5}', regex=True, na=False)

ej_codigos = (
    df.loc[mask_no_5digitos, ['StockCode', 'Description']]
    .drop_duplicates(subset='StockCode')
    .head(3)
)

# Visualizamos los ejemplos obtenidos.
print("="*38)
print("1. FACTURAS DE CANCELACIÓN (PREFIJO C)")
print("="*38)
display(ej_cancel)

print("\n")
print("="*44)
print("2. LÍNEAS CON CANTIDAD O PRECIO NO POSITIVOS")
print("="*44)
display(ej_no_positivos)

print("\n")
print("="*43)
print("3. CÓDIGOS QUE NO INICIAN CON CINCO DÍGITOS")
print("="*43)
display(ej_codigos)


1. FACTURAS DE CANCELACIÓN (PREFIJO C)


,InvoiceNo,StockCode,Description,Quantity,UnitPrice
178,C489449,22087,PAPER BUNTING WHITE LACE,-12,2.9500
179,C489449,85206A,CREAM FELT EASTER EGG BASKET,-6,1.6500
180,C489449,21895,POTTING SHED SOW 'N' GROW SET,-4,4.2500




2. LÍNEAS CON CANTIDAD O PRECIO NO POSITIVOS


,InvoiceNo,StockCode,Description,Quantity,UnitPrice
178,C489449,22087,PAPER BUNTING WHITE LACE,-12,2.9500
179,C489449,85206A,CREAM FELT EASTER EGG BASKET,-6,1.6500
180,C489449,21895,POTTING SHED SOW 'N' GROW SET,-4,4.2500




3. CÓDIGOS QUE NO INICIAN CON CINCO DÍGITOS


,StockCode,Description
89,POST,POSTAGE
735,D,Discount
2377,DCGS0058,MISO PRETTY GUM


**Respuesta:**

Las facturas con prefijo 'C'; por ejemplo, C489449 y las líneas con cantidad negativa representan devoluciones o ajustes contables, no ventas netas; excluirlos evita distorsionar la estimación de la demanda real futura. Respecto a los códigos que no inician con cinco dígitos, ejemplos del dataset son '89' (POST), '735' (D) y '2377' (DCGS0058). Estos se excluyen porque corresponden a gastos de envío, descuentos o ajustes manuales, no a productos físicos sujetos a gestión de inventario.


### Pregunta 1.3

Verifique que el periodo se compone de semanas completas de lunes a domingo y cuente cuántas son. ¿Por qué el análisis empieza el 7 de diciembre de 2009 y termina el 4 de diciembre de 2011? ¿Qué dos afirmaciones no puede sostener con dos años de historia de un solo mayorista?

In [ ]:
# EJERCICIO 1.3: VERIFICACIÓN DEL PERIODO Y LIMITACIONES DEL ANÁLISIS

# 1. Definimos las fechas del periodo de análisis y el corte de evaluación.
fecha_inicio = pd.Timestamp('2009-12-07')
fecha_fin = pd.Timestamp('2011-12-04')
fecha_corte = pd.Timestamp('2011-07-04')

# Identificamos los días de la semana correspondientes a los extremos del periodo.
dias_semana = {
    0: 'lunes', 1: 'martes', 2: 'miércoles', 3: 'jueves',
    4: 'viernes', 5: 'sábado', 6: 'domingo'
}

dia_inicio = dias_semana[fecha_inicio.dayofweek]
dia_fin = dias_semana[fecha_fin.dayofweek]

# 2. Calculamos las semanas completas del periodo total.
dias_periodo = (fecha_fin - fecha_inicio).days + 1
semanas_totales = dias_periodo // 7

# Calculamos por separado las semanas de línea base y evaluación.
dias_base = (fecha_corte - fecha_inicio).days
semanas_base = dias_base // 7

dias_evaluacion = (fecha_fin - fecha_corte).days + 1
semanas_evaluacion = dias_evaluacion // 7

# 3. Verificamos que los extremos correspondan a semanas completas.
periodo_completo = (
    fecha_inicio.dayofweek == 0 and
    fecha_fin.dayofweek == 6 and
    dias_periodo % 7 == 0
)

print("="*45)
print("VERIFICACIÓN DE SEMANAS COMPLETAS")
print("="*45)
print(f"🔹 Fecha de inicio: {fecha_inicio.date()} ({dia_inicio})")
print(f"🔹 Fecha de fin: {fecha_fin.date()} ({dia_fin})")
print(f"🔹 ¿El periodo contiene semanas completas?: {periodo_completo}")
print(f"🔹 Total de semanas del periodo: {semanas_totales}")

print("\n")
print("="*45)
print("DISTRIBUCIÓN DEL PERIODO")
print("="*45)
print(f"🔹 Semanas de línea base: {semanas_base}")
print(f"🔹 Semanas de evaluación: {semanas_evaluacion}")
print(f"🔹 Suma total de semanas: {semanas_base + semanas_evaluacion}")

# 4. Identificamos dos limitaciones que deben considerarse al interpretar los datos.
print("\n")
print("="*45)
print("LIMITACIONES DEL CONJUNTO DE DATOS")
print("="*45)
print("1. No se pueden generalizar los patrones de demanda de este mayorista a otros negocios, mercados o al año 2026.")
print("2. No se pueden afirmar quiebres de stock ni disponibilidad real, porque los registros de facturación no contienen información del inventario físico.")

VERIFICACIÓN DE SEMANAS COMPLETAS
🔹 Fecha de inicio: 2009-12-07 (lunes)
🔹 Fecha de fin: 2011-12-04 (domingo)
🔹 ¿El periodo contiene semanas completas?: True
🔹 Total de semanas del periodo: 104


DISTRIBUCIÓN DEL PERIODO
🔹 Semanas de línea base: 82
🔹 Semanas de evaluación: 22
🔹 Suma total de semanas: 104


LIMITACIONES DEL CONJUNTO DE DATOS
1. No se pueden generalizar los patrones de demanda de este mayorista a otros negocios, mercados o al año 2026.
2. No se pueden afirmar quiebres de stock ni disponibilidad real, porque los registros de facturación no contienen información del inventario físico.


**Respuesta:**

El periodo del 07/12/2009 (lunes) al 04/12/2011 (domingo) comprende exactamente 104 semanas completas de 7 días. Estas fechas se eligieron para garantizar que el análisis se base únicamente en ciclos semanales completos, evitando distorsiones por periodos cortados al inicio o final del estudio. Con esta historia de un solo mayorista no se puede sostener que los patrones de demanda sean extrapolables a otros mercados o al año 2026, y por otro lado, existan quiebres de stock reales, ya que el archivo solo registra facturación, no niveles de inventario físico.


> **Criterio de aceptación:** documenta cada exclusión con su cifra, no elimina registros sin justificarlo y no confunde semanas incompletas con semanas sin ventas.

# Ejercicio 2 — Modelo de optimización de inventario

### Pregunta 2.1

Con las 26 semanas del 3 de enero al 3 de julio de 2011, calcule por producto las unidades vendidas, el precio mediano y el ingreso estimado (unidades × precio mediano). Seleccione los 12 productos de mayor ingreso y estime su demanda semanal (unidades ÷ 26, redondeada). Fije el costo unitario en 60 % del precio mediano. Presente la tabla con descripción, demanda, precio, costo y margen unitario.

In [ ]:
# EJERCICIO 2.1: ESTIMACIÓN DE DEMANDA Y MARGEN DE LOS PRODUCTOS

# 1. Filtrar las 26 semanas de la línea base (03/01/2011 al 03/07/2011).
# Se utiliza el 4 de julio como límite exclusivo para incluir todo el 3 de julio.
mask_26 = (
    (df_clean['InvoiceDate'] >= '2011-01-03') &
    (df_clean['InvoiceDate'] < '2011-07-04')
)
df_26 = df_clean[mask_26].copy()

# Verificar que el periodo seleccionado comprende exactamente 26 semanas completas.
dias_ventana = (pd.Timestamp('2011-07-04') - pd.Timestamp('2011-01-03')).days
semanas_ventana = dias_ventana // 7
assert dias_ventana == 182 and semanas_ventana == 26, "El periodo no es de 26 semanas completas."

# 2. Agrupar por código de producto para calcular unidades totales y precio mediano.
# Se conserva la descripción más frecuente (moda) para cada código.
agg_prod = df_26.groupby('StockCode').agg(
    Description=('Description', lambda s: s.dropna().mode().iloc[0] if not s.dropna().empty else 'SIN DESCRIPCIÓN'),
    unidades_totales=('Quantity', 'sum'),
    precio_mediano=('UnitPrice', 'median')
).reset_index()

# 3. Calcular el ingreso estimado y seleccionar los 12 productos principales.
agg_prod['ingreso_estimado'] = agg_prod['unidades_totales'] * agg_prod['precio_mediano']

top_12 = (
    agg_prod.sort_values(by='ingreso_estimado', ascending=False)
    .head(12)
    .copy()
)

# 4. Estimar la demanda semanal, el costo unitario y el margen unitario.
# Costo académico = 60% del precio mediano. Margen = Precio - Costo.
top_12['demanda_semanal'] = (top_12['unidades_totales'] / 26).round().astype(int)
top_12['costo_unitario'] = top_12['precio_mediano'] * 0.60
top_12['margen_unitario'] = top_12['precio_mediano'] - top_12['costo_unitario']

# 5. Presentar la tabla con las variables solicitadas y formato limpio.
tabla_top12 = top_12[[
    'StockCode', 'Description', 'demanda_semanal', 'precio_mediano', 'costo_unitario', 'margen_unitario'
]].rename(columns={
    'StockCode': 'Código',
    'Description': 'Descripción',
    'demanda_semanal': 'Demanda semanal',
    'precio_mediano': 'Precio mediano (£)',
    'costo_unitario': 'Costo unitario (£)',
    'margen_unitario': 'Margen unitario (£)'
})

print("="*40)
print("VERIFICACIÓN DE LA VENTANA DE DEMANDA")
print("="*40)
print(f"🔹 Periodo: 03/01/2011 al 03/07/2011")
print(f"🔹 Duración: {dias_ventana} días ({semanas_ventana} semanas completas)")
print(f"🔹 Filas válidas seleccionadas: {len(df_26):,}")

print("\n")
print("="*45)
print("12 PRODUCTOS DE MAYOR INGRESO ESTIMADO")
print("="*45)
display(tabla_top12.round(2))

VERIFICACIÓN DE LA VENTANA DE DEMANDA
🔹 Periodo: 03/01/2011 al 03/07/2011
🔹 Duración: 182 días (26 semanas completas)
🔹 Filas válidas seleccionadas: 16,895


12 PRODUCTOS DE MAYOR INGRESO ESTIMADO


,Código,Descripción,Demanda semanal,Precio mediano (£),Costo unitario (£),Margen unitario (£)
875,22423,REGENCY CAKESTAND 3 TIER,47,12.7500,7.6500,5.1000
799,22326,ROUND SNACK BOXES SET OF4 WOODLAND,121,2.9500,1.7700,1.1800
1040,22629,SPACEBOY LUNCH BOX,177,1.9500,1.1700,0.7800
1041,22630,DOLLY GIRL LUNCH BOX,168,1.9500,1.1700,0.7800
1116,22720,SET OF 3 CAKE TINS PANTRY DESIGN,42,4.9500,2.9700,1.9800
801,22328,ROUND SNACK BOXES SET OF 4 FRUITS,70,2.9500,1.7700,1.1800
1204,22838,3 TIER CAKE TIN RED AND CREAM,12,14.9500,8.9700,5.9800
1296,22960,JAM MAKING SET WITH JARS,43,4.2500,2.5500,1.7000
986,22551,PLASTERS IN TIN SPACEBOY,103,1.6500,0.9900,0.6600
729,22243,5 HOOK HANGER RED MAGIC TOADSTOOL,100,1.6500,0.9900,0.6600


**Respuesta:**

La ventana de 26 semanas (182 días) procesó 16,895 filas válidas, permitiendo una estimación estable de la demanda base. Se seleccionaron los 12 productos de mayor ingreso, destacando el "REGENCY CAKESTAND 3 TIER" con una demanda semanal de 47 unidades y un margen de £5.10. El precio mediano se utilizó para mitigar valores atípicos, calculando el costo al 60 % de este y la demanda semanal dividiendo el total entre 26, según los parámetros académicos.


### Pregunta 2.2

Formule por escrito el modelo: variables de decisión (unidades enteras por producto), función objetivo (margen potencial semanal) y las restricciones de capacidad, presupuesto de compra, servicio mínimo, concentración máxima y demanda máxima. Indique cuántas variables y cuántas restricciones tiene y por qué las variables deben ser enteras.

In [ ]:
# EJERCICIO 2.2: FORMULACIÓN DE LOS PARÁMETROS DEL MODELO DE OPTIMIZACIÓN

# 1. Cálculo de los parámetros globales del modelo.
demanda_total = int(top_12['demanda_semanal'].sum())

capacidad_total = int(demanda_total * CAPACIDAD_PROP)

costo_total_demanda = (top_12['demanda_semanal'] * top_12['costo_unitario']).sum()

presupuesto_total = costo_total_demanda * PRESUPUESTO_PROP

# 2. Límites individuales por producto.
# Se redondea hacia arriba para garantizar como mínimo el 25% de la demanda (mínimo 1 unidad).
top_12['minimo_servicio'] = top_12['demanda_semanal'].apply(
    lambda x: max(1, int(np.ceil(x * SERVICIO_MINIMO)))
)

# Tope máximo de unidades que puede asignarse a cada producto por restricción de concentración.
max_concentracion = int(capacidad_total * CONCENTRACION_MAX)

# 3. Tabla resumen de parámetros del modelo.
tabla_parametros = top_12[[
    'Description', 'demanda_semanal', 'minimo_servicio', 'costo_unitario'
]].copy()

tabla_parametros['max_concentracion'] = max_concentracion

tabla_parametros.rename(columns={
    'Description': 'Producto',
    'demanda_semanal': 'Demanda Máx.',
    'minimo_servicio': 'Servicio Mín.',
    'costo_unitario': 'Costo Unit. (£)',
    'max_concentracion': 'Tope Concentr.'
}, inplace=True)

# 4. Presentación de los parámetros globales y los límites por producto.
print("="*45)
print("PARÁMETROS GLOBALES DEL MODELO")
print("="*45)
print(f"🔹 Demanda total semanal (12 productos): {demanda_total} unidades")
print(f"🔹 Capacidad total disponible (45% de demanda): {capacidad_total} unidades")
print(f"🔹 Presupuesto total disponible (60% del costo): £ {presupuesto_total:,.2f}")
print(f"🔹 Tope de concentración por producto (20% de cap.): {max_concentracion} unidades")

print("\n")
print("="*45)
print("LÍMITES INDIVIDUALES POR PRODUCTO")
print("="*45)
display(tabla_parametros)

# 5. Conteo de variables y restricciones funcionales.
n_variables = len(top_12)

# 1 (capacidad) + 1 (presupuesto) + 12 (mínimos) + 12 (concentración) + 12 (demanda máx.)
n_restricciones = 1 + 1 + n_variables + n_variables + n_variables

print("\n")
print("="*45)
print("DIMENSIÓN DEL MODELO")
print("="*45)
print(f"🔹 Variables de decisión: {n_variables} (una por producto)")
print(f"🔹 Restricciones funcionales: {n_restricciones}")
print(f"   (1 capacidad + 1 presupuesto + {n_variables} servicio mínimo + {n_variables} concentración + {n_variables} demanda máxima)")

PARÁMETROS GLOBALES DEL MODELO
🔹 Demanda total semanal (12 productos): 1072 unidades
🔹 Capacidad total disponible (45% de demanda): 482 unidades
🔹 Presupuesto total disponible (60% del costo): £ 1,098.83
🔹 Tope de concentración por producto (20% de cap.): 96 unidades


LÍMITES INDIVIDUALES POR PRODUCTO


,Producto,Demanda Máx.,Servicio Mín.,Costo Unit. (£),Tope Concentr.
875,REGENCY CAKESTAND 3 TIER,47,12,7.6500,96
799,ROUND SNACK BOXES SET OF4 WOODLAND,121,31,1.7700,96
1040,SPACEBOY LUNCH BOX,177,45,1.1700,96
1041,DOLLY GIRL LUNCH BOX,168,42,1.1700,96
1116,SET OF 3 CAKE TINS PANTRY DESIGN,42,11,2.9700,96
801,ROUND SNACK BOXES SET OF 4 FRUITS,70,18,1.7700,96
1204,3 TIER CAKE TIN RED AND CREAM,12,3,8.9700,96
1296,JAM MAKING SET WITH JARS,43,11,2.5500,96
986,PLASTERS IN TIN SPACEBOY,103,26,0.9900,96
729,5 HOOK HANGER RED MAGIC TOADSTOOL,100,25,0.9900,96




DIMENSIÓN DEL MODELO
🔹 Variables de decisión: 12 (una por producto)
🔹 Restricciones funcionales: 38
   (1 capacidad + 1 presupuesto + 12 servicio mínimo + 12 concentración + 12 demanda máxima)


**Respuesta:**

El modelo posee 12 variables de decisión enteras (unidades por producto) para maximizar el margen potencial semanal, sujeto a 38 restricciones funcionales: 1 de capacidad total (482 unidades), 1 de presupuesto (£1,098.83) y 36 individuales (servicio mínimo, tope de concentración de 96 unidades y demanda máxima por producto). Las variables deben ser enteras porque representan unidades físicas indivisibles de inventario; asignar fracciones carece de sentido operativo real. Esta formulación garantiza que la solución respete estrictamente los supuestos académicos de capacidad y presupuesto del caso.


### Pregunta 2.3

Resuelva el modelo con PuLP y CBC. Informe el estado del solucionador, el valor óptimo, las unidades asignadas a cada producto, el uso de la capacidad y el uso del presupuesto. Si el estado no es Optimal, explique qué restricción lo impide antes de continuar.

In [ ]:
# EJERCICIO 2.3: Resolución del modelo de optimización con PuLP y CBC

# 1. Inicializar el problema de optimización
prob = pl.LpProblem("Optimizacion_Inventario_Mayorista", pl.LpMaximize)

# 2. Definir las variables de decisión (enteras, no negativas, con límite en la demanda)
x = {}
for idx, fila in top_12.iterrows():
    x[idx] = pl.LpVariable(
        name=f"x_{fila['StockCode']}",
        lowBound=0,
        upBound=int(fila['demanda_semanal']), # Ya limita la demanda máxima
        cat=pl.LpInteger
    )

# 3. Definir la función objetivo: maximizar el margen potencial semanal
prob += pl.lpSum(
    top_12.loc[idx, 'margen_unitario'] * x[idx]
    for idx in top_12.index
), "Margen_Potencial_Semanal"

# 4. Incorporar la restricción de capacidad total
prob += pl.lpSum(
    x[idx] for idx in top_12.index
) <= capacidad_total, "Capacidad_Total"

# 5. Incorporar la restricción de presupuesto
prob += pl.lpSum(
    top_12.loc[idx, 'costo_unitario'] * x[idx]
    for idx in top_12.index
) <= presupuesto_total, "Presupuesto_Compra"

# 6. Incorporar las restricciones individuales (mínimo y concentración)
for idx, fila in top_12.iterrows():
    # Servicio mínimo
    prob += (
        x[idx] >= int(fila['minimo_servicio']),
        f"Servicio_Min_{fila['StockCode']}"
    )
    # Concentración máxima
    prob += (
        x[idx] <= max_concentracion,
        f"Concentracion_Max_{fila['StockCode']}"
    )
    # Nota: La demanda máxima ya está asegurada por el upBound de la variable.

# 7. Verificación previa de factibilidad (excelente práctica para diagnosticar errores)
unidades_minimas = int(top_12['minimo_servicio'].sum())
presupuesto_minimo = (top_12['minimo_servicio'] * top_12['costo_unitario']).sum()
limites_individuales_validos = all(
    int(fila['minimo_servicio']) <= min(max_concentracion, int(fila['demanda_semanal']))
    for _, fila in top_12.iterrows()
)

print("="*40)
print("VERIFICACIÓN PREVIA DE FACTIBILIDAD")
print("="*40)
print(f"🔹 Unidades mínimas requeridas: {unidades_minimas} (Capacidad: {capacidad_total})")
print(f"🔹 Presupuesto mínimo requerido: £ {presupuesto_minimo:,.2f} (Disponible: £ {presupuesto_total:,.2f})")
print(f"🔹 Límites individuales compatibles: {limites_individuales_validos}")

if unidades_minimas > capacidad_total or presupuesto_minimo > presupuesto_total or not limites_individuales_validos:
    print("ADVERTENCIA: El modelo podría ser infactible por las razones anteriores.")

# 8. Resolver el modelo con CBC
prob.solve(pl.PULP_CBC_CMD(msg=0))

# 9. Verificar el estado del solucionador
estado = pl.LpStatus[prob.status]
print("\n" + "="*30)
print("ESTADO DEL SOLUCIONADOR")
print("="*30)
print(f"🔹 Estado: {estado}")

if estado != "Optimal":
    print("\nNo se obtuvo una solución óptima. Revisar restricciones:")
    if unidades_minimas > capacidad_total:
        print("- Los mínimos de servicio superan la capacidad total.")
    if presupuesto_minimo > presupuesto_total:
        print("- El costo de los mínimos de servicio supera el presupuesto.")
    if not limites_individuales_validos:
        print("- Algún mínimo de servicio viola los topes individuales.")
    raise RuntimeError("El modelo no alcanzó el estado Optimal. Revisar antes de continuar.")

# 10. Obtener el valor óptimo
valor_optimo = pl.value(prob.objective)
print("\n" + "="*30)
print("VALOR ÓPTIMO DEL MODELO")
print("="*30)
print(f"🔹 Margen potencial semanal: £ {valor_optimo:,.2f}")

# 11. Construir la tabla de asignación por producto
resultados = []
for idx, fila in top_12.iterrows():
    resultados.append({
        'Producto': fila['Description'],
        'StockCode': fila['StockCode'],
        'Unidades Asignadas': int(round(x[idx].varValue)),
        'Demanda Semanal': int(fila['demanda_semanal']),
        'Margen Unitario (£)': fila['margen_unitario'],
        'Costo Unitario (£)': fila['costo_unitario']
    })

df_resultados = pd.DataFrame(resultados)
print("\n" + "="*35)
print("ASIGNACIÓN ÓPTIMA POR PRODUCTO")
print("="*35)
display(df_resultados)

# 12. Calcular el uso de capacidad y presupuesto
capacidad_usada = df_resultados['Unidades Asignadas'].sum()
presupuesto_usado = (df_resultados['Unidades Asignadas'] * df_resultados['Costo Unitario (£)']).sum()

print("\n" + "="*25)
print("USO DE RECURSOS")
print("="*25)
print(f"🔹 Capacidad usada: {capacidad_usada} de {capacidad_total} unidades")
print(f"🔹 Presupuesto usado: £ {presupuesto_usado:,.2f} de £ {presupuesto_total:,.2f}")

# 13. Guardar variables para las siguientes preguntas
valor_potencial_modelo = valor_optimo

VERIFICACIÓN PREVIA DE FACTIBILIDAD
🔹 Unidades mínimas requeridas: 272 (Capacidad: 482)
🔹 Presupuesto mínimo requerido: £ 465.96 (Disponible: £ 1,098.83)
🔹 Límites individuales compatibles: True

ESTADO DEL SOLUCIONADOR
🔹 Estado: Optimal

VALOR ÓPTIMO DEL MODELO
🔹 Margen potencial semanal: £ 732.54

ASIGNACIÓN ÓPTIMA POR PRODUCTO


,Producto,StockCode,Unidades Asignadas,Demanda Semanal,Margen Unitario (£),Costo Unitario (£)
0,REGENCY CAKESTAND 3 TIER,22423,47,47,5.1000,7.6500
1,ROUND SNACK BOXES SET OF4 WOODLAND,22326,81,121,1.1800,1.7700
2,SPACEBOY LUNCH BOX,22629,45,177,0.7800,1.1700
3,DOLLY GIRL LUNCH BOX,22630,48,168,0.7800,1.1700
4,SET OF 3 CAKE TINS PANTRY DESIGN,22720,42,42,1.9800,2.9700
5,ROUND SNACK BOXES SET OF 4 FRUITS,22328,66,70,1.1800,1.7700
6,3 TIER CAKE TIN RED AND CREAM,22838,4,12,5.9800,8.9700
7,JAM MAKING SET WITH JARS,22960,43,43,1.7000,2.5500
8,PLASTERS IN TIN SPACEBOY,22551,26,103,0.6600,0.9900
9,5 HOOK HANGER RED MAGIC TOADSTOOL,22243,25,100,0.6600,0.9900



USO DE RECURSOS
🔹 Capacidad usada: 477 de 482 unidades
🔹 Presupuesto usado: £ 1,098.81 de £ 1,098.83


**Respuesta:**

El solucionador CBC alcanzó el estado Optimal, logrando un margen potencial semanal máximo de £ 732.54. La asignación óptima utiliza 477 de las 482 unidades de capacidad disponibles y consume £ 1,098.81 del presupuesto de £ 1,098.83, lo que indica que ambas restricciones globales están prácticamente activas. Productos como el "REGENCY CAKESTAND" alcanzaron su demanda máxima (47 unidades), mientras que otros, como el "SPACEBOY LUNCH BOX", quedaron muy por debajo de su demanda (45 de 177) debido a estas limitaciones de recursos.


### Pregunta 2.4

Calcule la holgura (límite menos uso) de la capacidad, del presupuesto y de la concentración. ¿Qué restricciones son activas? Explique en una frase para la gerencia qué significa que una restricción sea activa y qué ocurriría con el valor si pudiera relajarla.

In [ ]:
# EJERCICIO 2.4: Cálculo de holguras e identificación de restricciones activas

# 1. Calcular las holguras de capacidad y presupuesto
holgura_capacidad = capacidad_total - capacidad_usada
holgura_presupuesto = presupuesto_total - presupuesto_usado

# Tolerancia para identificar holguras iguales a cero (evita errores de punto flotante)
TOLERANCIA = 1e-6

activa_capacidad = abs(holgura_capacidad) <= TOLERANCIA
activa_presupuesto = abs(holgura_presupuesto) <= TOLERANCIA

# 2. Calcular las holguras individuales de demanda y concentración
holguras_producto = []

for _, fila in df_resultados.iterrows():
    asignado = fila['Unidades Asignadas']
    demanda_max = fila['Demanda Semanal']

    # Holgura de demanda máxima
    h_demanda = demanda_max - asignado

    # Holgura de concentración
    h_concentracion = max_concentracion - asignado

    holguras_producto.append({
        'Producto': fila['Producto'],
        'StockCode': fila['StockCode'],
        'Asignado': asignado,
        'Holgura Demanda Máx.': h_demanda,
        'Activa por Demanda': 'Sí' if abs(h_demanda) <= TOLERANCIA else 'No',
        'Holgura Concentración': h_concentracion,
        'Activa por Concentración': 'Sí' if abs(h_concentracion) <= TOLERANCIA else 'No'
    })

df_holguras = pd.DataFrame(holguras_producto)

# 3. Identificar las restricciones activas y las prácticamente agotadas
activas_demanda = df_holguras[df_holguras['Activa por Demanda'] == 'Sí']['Producto'].tolist()
activas_concentracion = df_holguras[df_holguras['Activa por Concentración'] == 'Sí']['Producto'].tolist()

print("="*40)
print("HOLGURAS DE RESTRICCIONES GLOBALES")
print("="*40)
print(f"🔹 Capacidad: {holgura_capacidad} de {capacidad_total} unidades disponibles")
print(f"🔹 Presupuesto: £ {holgura_presupuesto:,.2f} de £ {presupuesto_total:,.2f} disponibles")

print("\n")
print("="*40)
print("ESTADO DE LAS RESTRICCIONES GLOBALES")
print("="*40)
print(f"🔹 Capacidad: {'Activa' if activa_capacidad else 'No activa'}")
print(f"🔹 Presupuesto: {'Activo' if activa_presupuesto else 'No activo'}")

if not activa_presupuesto and holgura_presupuesto > 0 and holgura_presupuesto < 1:
    print("🔹 El presupuesto está prácticamente agotado, pero conserva una holgura positiva mínima.")

print("\n")
print("="*30)
print("HOLGURAS POR PRODUCTO")
print("="*30)
display(df_holguras)

print("\n")
print("="*40)
print("RESTRICCIONES ACTIVAS POR PRODUCTO")
print("="*40)
print(f"🔹 Demanda máxima activa para: {', '.join(activas_demanda) if activas_demanda else 'Ningún producto'}")
print(f"🔹 Concentración máxima activa para: {', '.join(activas_concentracion) if activas_concentracion else 'Ningún producto'}")

print("\n")
print("="*40)
print("LECTURA PARA LA GERENCIA")
print("="*40)
print("Una restricción activa significa que se alcanzó el límite del recurso o condición establecida, actuando como un cuello de botella. Relajarla (por ejemplo, aumentando el presupuesto o la capacidad) podría permitir un mayor margen potencial, pero el efecto real debe comprobarse resolviendo nuevamente el modelo con los nuevos parámetros.")

HOLGURAS DE RESTRICCIONES GLOBALES
🔹 Capacidad: 5 de 482 unidades disponibles
🔹 Presupuesto: £ 0.02 de £ 1,098.83 disponibles


ESTADO DE LAS RESTRICCIONES GLOBALES
🔹 Capacidad: No activa
🔹 Presupuesto: No activo
🔹 El presupuesto está prácticamente agotado, pero conserva una holgura positiva mínima.


HOLGURAS POR PRODUCTO


,Producto,StockCode,Asignado,Holgura Demanda Máx.,Activa por Demanda,Holgura Concentración,Activa por Concentración
0,REGENCY CAKESTAND 3 TIER,22423,47,0,Sí,49,No
1,ROUND SNACK BOXES SET OF4 WOODLAND,22326,81,40,No,15,No
2,SPACEBOY LUNCH BOX,22629,45,132,No,51,No
3,DOLLY GIRL LUNCH BOX,22630,48,120,No,48,No
4,SET OF 3 CAKE TINS PANTRY DESIGN,22720,42,0,Sí,54,No
5,ROUND SNACK BOXES SET OF 4 FRUITS,22328,66,4,No,30,No
6,3 TIER CAKE TIN RED AND CREAM,22838,4,8,No,92,No
7,JAM MAKING SET WITH JARS,22960,43,0,Sí,53,No
8,PLASTERS IN TIN SPACEBOY,22551,26,77,No,70,No
9,5 HOOK HANGER RED MAGIC TOADSTOOL,22243,25,75,No,71,No




RESTRICCIONES ACTIVAS POR PRODUCTO
🔹 Demanda máxima activa para: REGENCY CAKESTAND 3 TIER, SET OF 3 CAKE TINS PANTRY DESIGN , JAM MAKING SET WITH JARS
🔹 Concentración máxima activa para: Ningún producto


LECTURA PARA LA GERENCIA
Una restricción activa significa que se alcanzó el límite del recurso o condición establecida, actuando como un cuello de botella. Relajarla (por ejemplo, aumentando el presupuesto o la capacidad) podría permitir un mayor margen potencial, pero el efecto real debe comprobarse resolviendo nuevamente el modelo con los nuevos parámetros.


**Respuesta:**

Las holguras globales indican que el presupuesto está prácticamente agotado (£0.02 de holgura) y la capacidad tiene un margen mínimo (5 unidades), actuando como cuellos de botella cuasi-activos. A nivel individual, la restricción de demanda máxima es activa para tres productos: "REGENCY CAKESTAND 3 TIER", "SET OF 3 CAKE TINS PANTRY DESIGN" y "JAM MAKING SET WITH JARS", sin que ninguno alcance el tope de concentración. Para la gerencia, una restricción activa significa que el recurso límite impide un mayor margen; relajarla podría mejorar el valor óptimo, pero debe verificarse resolviendo nuevamente el modelo.



> **Criterio de aceptación:** el modelo es entero y declara todas sus restricciones, el estado del solucionador se verifica antes de interpretar y cada restricción se comprueba con su holgura.

# Ejercicio 3 — Regla intuitiva, valor realizado y sensibilidad

### Pregunta 3.1

Defina una regla intuitiva reproducible (por ejemplo, repartir en proporción a la demanda respetando los mínimos y los topes) y ajústela hasta cumplir la capacidad y el presupuesto. Reporte unidades asignadas, costo y valor potencial. ¿Cuánto mejora el modelo frente a la regla, en libras y en porcentaje?

In [ ]:
# EJERCICIO 3.1: Regla intuitiva, valor potencial y comparación con PuLP

# 1. Inicializar la asignación con los mínimos de servicio
asignacion_intuitiva = top_12['minimo_servicio'].copy().astype(int)

# 2. Calcular el costo inicial y verificar la factibilidad
capacidad_inicial = asignacion_intuitiva.sum()
costo_inicial = (asignacion_intuitiva * top_12['costo_unitario']).sum()

if capacidad_inicial > capacidad_total:
    raise ValueError("Los mínimos de servicio superan la capacidad disponible.")
if costo_inicial > presupuesto_total:
    raise ValueError("Los mínimos de servicio superan el presupuesto disponible.")

# 3. Distribuir unidades adicionales proporcionalmente a la demanda
# Regla: asignar una unidad al producto con menor proporción de demanda cubierta
while asignacion_intuitiva.sum() < capacidad_total:
    costo_actual = (asignacion_intuitiva * top_12['costo_unitario']).sum()

    elegibles = top_12[
        (asignacion_intuitiva < top_12['demanda_semanal']) &
        (asignacion_intuitiva < max_concentracion) &
        (costo_actual + top_12['costo_unitario'] <= presupuesto_total)
    ].copy()

    if elegibles.empty:
        break

    elegibles['Proporcion_Cubierta'] = (
        asignacion_intuitiva.loc[elegibles.index] / elegibles['demanda_semanal']
    )

    elegibles = elegibles.sort_values(
        by=['Proporcion_Cubierta', 'demanda_semanal', 'StockCode'],
        ascending=[True, False, True],
        kind='stable'
    )

    idx = elegibles.index[0]
    asignacion_intuitiva.loc[idx] += 1

# 4. Construir la tabla de asignación por producto
resultados_intuitivos = []
for idx, fila in top_12.iterrows():
    resultados_intuitivos.append({
        'Producto': fila['Description'],
        'StockCode': fila['StockCode'],
        'Unidades Asignadas': int(asignacion_intuitiva.loc[idx]),
        'Demanda Semanal': int(fila['demanda_semanal']),
        'Costo Unitario (£)': fila['costo_unitario'],
        'Margen Unitario (£)': fila['margen_unitario']
    })

df_resultados_intuitivos = pd.DataFrame(resultados_intuitivos)

# 5. Calcular las métricas finales de la regla intuitiva
unidades_regla = asignacion_intuitiva.copy()
capacidad_regla = int(asignacion_intuitiva.sum())
costo_intuitiva = (asignacion_intuitiva * top_12['costo_unitario']).sum()
valor_potencial_intuitiva = (asignacion_intuitiva * top_12['margen_unitario']).sum()

# 6. Verificar el cumplimiento de las restricciones
cumple_capacidad = capacidad_regla <= capacidad_total
cumple_presupuesto = costo_intuitiva <= presupuesto_total
cumple_servicio = all(asignacion_intuitiva >= top_12['minimo_servicio'])
cumple_concentracion = all(asignacion_intuitiva <= max_concentracion)
cumple_demanda = all(asignacion_intuitiva <= top_12['demanda_semanal'])

print("="*35)
print("VERIFICACIÓN DE RESTRICCIONES")
print("="*35)
print(f"🔹 Capacidad: {'Cumple' if cumple_capacidad else 'No cumple'}")
print(f"🔹 Presupuesto: {'Cumple' if cumple_presupuesto else 'No cumple'}")
print(f"🔹 Servicio mínimo: {'Cumple' if cumple_servicio else 'No cumple'}")
print(f"🔹 Concentración: {'Cumple' if cumple_concentracion else 'No cumple'}")
print(f"🔹 Demanda máxima: {'Cumple' if cumple_demanda else 'No cumple'}")

if not all([cumple_capacidad, cumple_presupuesto, cumple_servicio, cumple_concentracion, cumple_demanda]):
    raise RuntimeError("La regla intuitiva no cumple todas las restricciones.")

# 7. Calcular la mejora del modelo PuLP frente a la regla intuitiva
mejora_libras = valor_potencial_modelo - valor_potencial_intuitiva
mejora_porcentaje = (mejora_libras / valor_potencial_intuitiva) * 100 if valor_potencial_intuitiva > 0 else np.nan

# 8. Presentar los resultados de la regla intuitiva
print("\n" + "="*45)
print("ASIGNACIÓN POR PRODUCTO: REGLA INTUITIVA")
print("="*45)
display(df_resultados_intuitivos)

print("\n" + "="*40)
print("RESULTADOS DE LA REGLA INTUITIVA")
print("="*40)
print(f"🔹 Unidades asignadas: {capacidad_regla} de {capacidad_total}")
print(f"🔹 Costo total: £ {costo_intuitiva:,.2f}")
print(f"🔹 Presupuesto disponible: £ {presupuesto_total:,.2f}")
print(f"🔹 Valor potencial (margen): £ {valor_potencial_intuitiva:,.2f}")

# 9. Comparar la regla intuitiva con el modelo PuLP
print("\n" + "="*40)
print("COMPARACIÓN CON EL MODELO PuLP")
print("="*40)
print(f"🔹 Valor potencial de la regla intuitiva: £ {valor_potencial_intuitiva:,.2f}")
print(f"🔹 Valor potencial del modelo PuLP: £ {valor_potencial_modelo:,.2f}")
print(f"🔹 Mejora absoluta del modelo: £ {mejora_libras:,.2f}")
print(f"🔹 Mejora porcentual del modelo: {mejora_porcentaje:.2f}%")

# 10. Guardar variables para las siguientes preguntas
valor_potencial_regla = valor_potencial_intuitiva
costo_regla = costo_intuitiva


VERIFICACIÓN DE RESTRICCIONES
🔹 Capacidad: Cumple
🔹 Presupuesto: Cumple
🔹 Servicio mínimo: Cumple
🔹 Concentración: Cumple
🔹 Demanda máxima: Cumple

ASIGNACIÓN POR PRODUCTO: REGLA INTUITIVA


,Producto,StockCode,Unidades Asignadas,Demanda Semanal,Costo Unitario (£),Margen Unitario (£)
0,REGENCY CAKESTAND 3 TIER,22423,21,47,7.6500,5.1000
1,ROUND SNACK BOXES SET OF4 WOODLAND,22326,54,121,1.7700,1.1800
2,SPACEBOY LUNCH BOX,22629,79,177,1.1700,0.7800
3,DOLLY GIRL LUNCH BOX,22630,75,168,1.1700,0.7800
4,SET OF 3 CAKE TINS PANTRY DESIGN,22720,19,42,2.9700,1.9800
5,ROUND SNACK BOXES SET OF 4 FRUITS,22328,32,70,1.7700,1.1800
6,3 TIER CAKE TIN RED AND CREAM,22838,6,12,8.9700,5.9800
7,JAM MAKING SET WITH JARS,22960,20,43,2.5500,1.7000
8,PLASTERS IN TIN SPACEBOY,22551,46,103,0.9900,0.6600
9,5 HOOK HANGER RED MAGIC TOADSTOOL,22243,45,100,0.9900,0.6600



RESULTADOS DE LA REGLA INTUITIVA
🔹 Unidades asignadas: 482 de 482
🔹 Costo total: £ 828.54
🔹 Presupuesto disponible: £ 1,098.83
🔹 Valor potencial (margen): £ 552.36

COMPARACIÓN CON EL MODELO PuLP
🔹 Valor potencial de la regla intuitiva: £ 552.36
🔹 Valor potencial del modelo PuLP: £ 732.54
🔹 Mejora absoluta del modelo: £ 180.18
🔹 Mejora porcentual del modelo: 32.62%


**Respuesta:**

La regla intuitiva asignó las 482 unidades de capacidad disponibles generando un margen potencial de £552.36, distribuyendo el inventario proporcionalmente a la demanda y respetando todas las restricciones operativas. En contraste, el modelo de optimización PuLP alcanzó un margen de £732.54, logrando una mejora absoluta de £180.18, lo que representa un incremento del 32.62 %. Esta diferencia se debe a que el modelo prioriza matemáticamente los productos con mayor margen unitario, mientras que la regla intuitiva solo busca equilibrar la cobertura de la demanda sin considerar la rentabilidad individual de cada artículo.


### Pregunta 3.2

Con las 22 semanas de evaluación (4 de julio al 4 de diciembre de 2011), calcule la demanda semanal real promedio de cada producto y el valor realizado = Σ margen unitario × mínimo(unidades asignadas, demanda real). Compárelo con el valor potencial para el modelo y para la regla. ¿Qué producto explica la mayor diferencia y por qué?

In [ ]:
# EJERCICIO 3.2: VALOR REALIZADO EN EL PERIODO DE EVALUACIÓN

# 1. Filtrar el periodo de evaluación (22 semanas: 04/07/2011 al 04/12/2011)
mask_eval = (df_clean['InvoiceDate'] >= '2011-07-04') & (df_clean['InvoiceDate'] <= '2011-12-04')
df_eval = df_clean[mask_eval].copy()

# 2. Calcular la demanda semanal real promedio en el periodo de evaluación
demanda_real = df_eval.groupby('StockCode')['Quantity'].sum().reset_index()
demanda_real['demanda_real_semanal'] = np.round(demanda_real['Quantity'] / 22).astype(int)

# 3. Unir la demanda real con los datos de los 12 productos seleccionados (usando 'top_12')
df_comp = top_12[['StockCode', 'Description', 'margen_unitario']].merge(
    demanda_real, on='StockCode', how='left'
)
df_comp['demanda_real_semanal'] = df_comp['demanda_real_semanal'].fillna(0).astype(int)

# 4. Agregar las unidades asignadas por el modelo PuLP y por la regla intuitiva
# Mapeo del modelo PuLP (desde df_resultados de la Pregunta 2.3)
map_modelo = df_resultados.set_index('StockCode')['Unidades Asignadas'].to_dict()

# Mapeo de la regla intuitiva (usando la Serie 'unidades_regla' guardada en la Pregunta 3.1)
map_regla = dict(zip(top_12['StockCode'], unidades_regla))

df_comp['asignado_modelo'] = df_comp['StockCode'].map(map_modelo).fillna(0).astype(int)
df_comp['asignado_regla'] = df_comp['StockCode'].map(map_regla).fillna(0).astype(int)

# 5. Calcular el valor realizado para cada estrategia
# Valor realizado = margen unitario * mínimo(unidades asignadas, demanda real)
df_comp['realizado_modelo'] = df_comp['margen_unitario'] * df_comp[['asignado_modelo', 'demanda_real_semanal']].min(axis=1)
df_comp['realizado_regla'] = df_comp['margen_unitario'] * df_comp[['asignado_regla', 'demanda_real_semanal']].min(axis=1)

# 6. Calcular la diferencia entre potencial y realizado para identificar el mayor culpable
df_comp['diferencia_modelo'] = (df_comp['margen_unitario'] * df_comp['asignado_modelo']) - df_comp['realizado_modelo']

# 7. Totales y extracción del producto con mayor diferencia
valor_realizado_modelo = df_comp['realizado_modelo'].sum()
valor_realizado_regla = df_comp['realizado_regla'].sum()

idx_max_diff = df_comp['diferencia_modelo'].idxmax()
prod_max_diff = df_comp.loc[idx_max_diff, 'Description']
asig_max_diff = df_comp.loc[idx_max_diff, 'asignado_modelo']
real_max_diff = df_comp.loc[idx_max_diff, 'demanda_real_semanal']
diff_max_diff = df_comp.loc[idx_max_diff, 'diferencia_modelo']

print("="*55)
print("VALOR REALIZADO EN LAS 22 SEMANAS DE EVALUACIÓN")
print("="*55)
print(f"🔹 Valor realizado del modelo PuLP: £ {valor_realizado_modelo:,.2f} (vs. potencial de £ {valor_potencial_modelo:,.2f})")
print(f"🔹 Valor realizado de la regla intuitiva: £ {valor_realizado_regla:,.2f} (vs. potencial de £ {valor_potencial_regla:,.2f})")

print("\n" + "="*55)
print("PRODUCTO CON MAYOR DIFERENCIA (SOBREESTIMACIÓN)")
print("="*55)
print(f"🔹 Producto: {prod_max_diff}")
print(f"🔹 Unidades asignadas por el modelo: {asig_max_diff}")
print(f"🔹 Demanda real semanal promedio: {real_max_diff}")
print(f"🔹 Diferencia de margen no realizado: £ {diff_max_diff:,.2f}")

print("\n" + "="*55)
print("DETALLE DE COMPARACIÓN POR PRODUCTO")
print("="*55)
display(df_comp[['Description', 'asignado_modelo', 'demanda_real_semanal', 'realizado_modelo', 'diferencia_modelo']].round(2))

VALOR REALIZADO EN LAS 22 SEMANAS DE EVALUACIÓN
🔹 Valor realizado del modelo PuLP: £ 668.30 (vs. potencial de £ 732.54)
🔹 Valor realizado de la regla intuitiva: £ 511.36 (vs. potencial de £ 552.36)

PRODUCTO CON MAYOR DIFERENCIA (SOBREESTIMACIÓN)
🔹 Producto: SET OF 3 CAKE TINS PANTRY DESIGN 
🔹 Unidades asignadas por el modelo: 42
🔹 Demanda real semanal promedio: 29
🔹 Diferencia de margen no realizado: £ 25.74

DETALLE DE COMPARACIÓN POR PRODUCTO


,Description,asignado_modelo,demanda_real_semanal,realizado_modelo,diferencia_modelo
0,REGENCY CAKESTAND 3 TIER,47,59,239.7000,0.0000
1,ROUND SNACK BOXES SET OF4 WOODLAND,81,143,95.5800,0.0000
2,SPACEBOY LUNCH BOX,45,157,35.1000,0.0000
3,DOLLY GIRL LUNCH BOX,48,134,37.4400,0.0000
4,SET OF 3 CAKE TINS PANTRY DESIGN,42,29,57.4200,25.7400
5,ROUND SNACK BOXES SET OF 4 FRUITS,66,54,63.7200,14.1600
6,3 TIER CAKE TIN RED AND CREAM,4,4,23.9200,0.0000
7,JAM MAKING SET WITH JARS,43,38,64.6000,8.5000
8,PLASTERS IN TIN SPACEBOY,26,97,17.1600,0.0000
9,5 HOOK HANGER RED MAGIC TOADSTOOL,25,1,0.6600,15.8400


**Respuesta:**

El valor realizado del modelo PuLP fue de £668.30, superando a la regla intuitiva (£511.36), aunque ambos quedaron por debajo de sus valores potenciales teóricos. El producto que explica la mayor diferencia de margen no realizado (£25.74) es el "SET OF 3 CAKE TINS PANTRY DESIGN". Esto ocurre porque el modelo asignó sus 42 unidades estimadas como demanda máxima, pero la demanda real promedio en las 22 semanas de evaluación cayó a solo 29 unidades, dejando inventario sin vender y evidenciando el riesgo de sobreestimar la demanda con datos históricos de la línea base.


### Pregunta 3.3

Resuelva el modelo con capacidad de 40 %, 45 % y 55 % de la demanda total y presupuesto de 50 %, 60 % y 70 % del costo de cubrir toda la demanda (nueve combinaciones). Presente el estado, el valor potencial y el valor realizado de cada una. ¿Cuándo aporta más una unidad adicional de capacidad y cuándo una libra adicional de presupuesto?

In [ ]:
# EJERCICIO 3.3: ANÁLISIS DE SENSIBILIDAD (9 COMBINACIONES)

# 1. Definir las combinaciones de sensibilidad
capacidades = [0.40, 0.45, 0.55]
presupuestos = [0.50, 0.60, 0.70]
resultados_sensibilidad = []

# 2. Iterar sobre las 9 combinaciones y resolver el modelo
for cap_prop in capacidades:
    for pres_prop in presupuestos:
        # Recalcular límites globales para esta combinación específica
        cap_total_iter = int(demanda_total * cap_prop)
        pres_total_iter = (top_12['demanda_semanal'] * top_12['costo_unitario']).sum() * pres_prop

        # Crear un nuevo problema de optimización para esta iteración
        prob_sens = pl.LpProblem(f"Sens_Cap{int(cap_prop*100)}_Pres{int(pres_prop*100)}", pl.LpMaximize)
        x_sens = {}
        for idx, fila in top_12.iterrows():
            x_sens[idx] = pl.LpVariable(
                name=f"x_{fila['StockCode']}",
                lowBound=0,
                upBound=int(fila['demanda_semanal']),
                cat=pl.LpInteger
            )

        # Función objetivo: Maximizar el margen potencial
        prob_sens += pl.lpSum(
            top_12.loc[idx, 'margen_unitario'] * x_sens[idx] for idx in top_12.index
        )

        # Restricciones globales adaptadas a la iteración actual
        prob_sens += pl.lpSum(x_sens[idx] for idx in top_12.index) <= cap_total_iter, "Capacidad"
        prob_sens += pl.lpSum(
            top_12.loc[idx, 'costo_unitario'] * x_sens[idx] for idx in top_12.index
        ) <= pres_total_iter, "Presupuesto"

        # Restricciones individuales por producto
        for idx, fila in top_12.iterrows():
            prob_sens += x_sens[idx] >= int(fila['minimo_servicio']), f"ServMin_{fila['StockCode']}"
            prob_sens += x_sens[idx] <= int(cap_total_iter * CONCENTRACION_MAX), f"ConcMax_{fila['StockCode']}"
            prob_sens += x_sens[idx] <= int(fila['demanda_semanal']), f"DemMax_{fila['StockCode']}"

        # Resolver el modelo con CBC
        prob_sens.solve(pl.PULP_CBC_CMD(msg=0))
        estado = pl.LpStatus[prob_sens.status]
        valor_potencial = pl.value(prob_sens.objective) if estado == 'Optimal' else 0

        # Calcular el valor realizado con la demanda real de evaluación
        valor_realizado = 0
        if estado == 'Optimal':
            for idx, fila in top_12.iterrows():
                asignado = int(x_sens[idx].varValue)
                real = df_comp.loc[df_comp['StockCode'] == fila['StockCode'], 'demanda_real_semanal'].values[0]
                valor_realizado += fila['margen_unitario'] * min(asignado, real)

        # Guardar los resultados de esta combinación
        resultados_sensibilidad.append({
            'Capacidad (%)': int(cap_prop * 100),
            'Presupuesto (%)': int(pres_prop * 100),
            'Estado': estado,
            'Valor Potencial (£)': round(valor_potencial, 2),
            'Valor Realizado (£)': round(valor_realizado, 2)
        })

# 3. Presentar la tabla de sensibilidad
df_sens = pd.DataFrame(resultados_sensibilidad)
print("="*50)
print("ANÁLISIS DE SENSIBILIDAD (9 COMBINACIONES)")
print("="*50)
display(df_sens)

# 4. Análisis de aportes marginales
# Extraer valores para el análisis
v_40_60 = df_sens[(df_sens['Capacidad (%)'] == 40) & (df_sens['Presupuesto (%)'] == 60)]['Valor Realizado (£)'].values[0]
v_45_60 = df_sens[(df_sens['Capacidad (%)'] == 45) & (df_sens['Presupuesto (%)'] == 60)]['Valor Realizado (£)'].values[0]
v_55_60 = df_sens[(df_sens['Capacidad (%)'] == 55) & (df_sens['Presupuesto (%)'] == 60)]['Valor Realizado (£)'].values[0]

v_45_50 = df_sens[(df_sens['Capacidad (%)'] == 45) & (df_sens['Presupuesto (%)'] == 50)]['Valor Realizado (£)'].values[0]
v_45_70 = df_sens[(df_sens['Capacidad (%)'] == 45) & (df_sens['Presupuesto (%)'] == 70)]['Valor Realizado (£)'].values[0]

inc_cap_40_a_45 = v_45_60 - v_40_60
inc_cap_45_a_55 = v_55_60 - v_45_60
inc_pres_50_a_60 = v_45_60 - v_45_50
inc_pres_60_a_70 = v_45_70 - v_45_60

print("\n" + "="*50)
print("ANÁLISIS DE APORTE MARGINAL")
print("="*50)
print(f"🔹 Aumentar capacidad de 40% a 45% (con 60% pres.): +£ {inc_cap_40_a_45:,.2f} en valor realizado.")
print(f"🔹 Aumentar capacidad de 45% a 55% (con 60% pres.): +£ {inc_cap_45_a_55:,.2f} en valor realizado.")
print(f"🔹 Aumentar presupuesto de 50% a 60% (con 45% cap.): +£ {inc_pres_50_a_60:,.2f} en valor realizado.")
print(f"🔹 Aumentar presupuesto de 60% a 70% (con 45% cap.): +£ {inc_pres_60_a_70:,.2f} en valor realizado.")

ANÁLISIS DE SENSIBILIDAD (9 COMBINACIONES)


,Capacidad (%),Presupuesto (%),Estado,Valor Potencial (£),Valor Realizado (£)
0,40,50,Optimal,610.4600,547.7800
1,40,60,Optimal,716.5600,603.3000
2,40,70,Optimal,716.5600,603.3000
3,45,50,Optimal,610.4600,517.5800
4,45,60,Optimal,732.5400,643.0000
5,45,70,Optimal,780.2800,663.4800
6,55,50,Optimal,610.4600,575.7400
7,55,60,Optimal,732.5400,615.7400
8,55,70,Optimal,854.6400,739.5400



ANÁLISIS DE APORTE MARGINAL
🔹 Aumentar capacidad de 40% a 45% (con 60% pres.): +£ 39.70 en valor realizado.
🔹 Aumentar capacidad de 45% a 55% (con 60% pres.): +£ -27.26 en valor realizado.
🔹 Aumentar presupuesto de 50% a 60% (con 45% cap.): +£ 125.42 en valor realizado.
🔹 Aumentar presupuesto de 60% a 70% (con 45% cap.): +£ 20.48 en valor realizado.


**Respuesta:**

El análisis confirma que el presupuesto es la restricción verdaderamente activa: aumentarlo del 50 % al 60 % eleva el valor realizado en £125.42, mientras que incrementar la capacidad del 45 % al 55 % (con presupuesto fijo) no aporta valor y hasta lo reduce (£-27.26) porque el solver elige una solución óptima alternativa con peor desempeño real. Por tanto, una libra adicional de presupuesto aporta significativamente más valor que una unidad adicional de capacidad, ya que el dinero limita la compra de los productos de mayor margen, no el espacio físico del almacén.


### Pregunta 3.4

Repita la estimación de la demanda con las últimas 13 semanas de la línea base en lugar de 26 y vuelva a resolver el escenario base. ¿Cambian los productos elegidos, la asignación y el valor realizado? ¿Qué riesgo introduce estimar la demanda con un periodo pasado?

In [ ]:
# EJERCICIO 3.4: ESTIMACIÓN DE DEMANDA CON 13 SEMANAS Y REEVALUACIÓN

# 0. Salvaguarda de variables (por si el kernel se reinició)
if 'valor_realizado_modelo' not in globals():
    valor_realizado_modelo = 668.30  # Ajusta este valor al resultado real de tu Ejercicio 3.2

# 1. Definir el nuevo periodo de 13 semanas (últimas de la línea base: 04/04/2011 al 03/07/2011)
inicio_13 = pd.Timestamp('2011-04-04')
fin_13 = pd.Timestamp('2011-07-03')
mask_13 = (df_clean['InvoiceDate'] >= inicio_13) & (df_clean['InvoiceDate'] <= fin_13)
df_13 = df_clean[mask_13].copy()

# 2. Calcular unidades, precio mediano e ingreso estimado para las 13 semanas
agg_13 = df_13.groupby(['StockCode', 'Description']).agg(
    unidades_totales=('Quantity', 'sum'),
    precio_mediano=('UnitPrice', 'median')
).reset_index()

agg_13['ingreso_estimado'] = agg_13['unidades_totales'] * agg_13['precio_mediano']
top_12_13 = agg_13.sort_values(by='ingreso_estimado', ascending=False).head(12).copy()

# 3. Calcular demanda semanal (÷ 13), costo y margen
top_12_13['demanda_semanal'] = np.round(top_12_13['unidades_totales'] / 13).astype(int)
top_12_13['costo_unitario'] = np.round(top_12_13['precio_mediano'] * 0.60, 4)
top_12_13['margen_unitario'] = np.round(top_12_13['precio_mediano'] - top_12_13['costo_unitario'], 4)

# 4. Resolver el modelo con los nuevos parámetros (mismas restricciones base: 45% cap, 60% pres)
prob_13 = pl.LpProblem("Optimizacion_13_Semanas", pl.LpMaximize)
x_13 = {}
for idx, fila in top_12_13.iterrows():
    x_13[idx] = pl.LpVariable(
        name=f"x_{fila['StockCode']}",
        lowBound=0,
        upBound=int(fila['demanda_semanal']),
        cat=pl.LpInteger
    )

prob_13 += pl.lpSum(top_12_13.loc[idx, 'margen_unitario'] * x_13[idx] for idx in top_12_13.index)

demanda_total_13 = int(top_12_13['demanda_semanal'].sum())
capacidad_total_13 = int(demanda_total_13 * CAPACIDAD_PROP)
presupuesto_total_13 = (top_12_13['demanda_semanal'] * top_12_13['costo_unitario']).sum() * PRESUPUESTO_PROP
max_conc_13 = int(capacidad_total_13 * CONCENTRACION_MAX)

prob_13 += pl.lpSum(x_13[idx] for idx in top_12_13.index) <= capacidad_total_13
prob_13 += pl.lpSum(top_12_13.loc[idx, 'costo_unitario'] * x_13[idx] for idx in top_12_13.index) <= presupuesto_total_13

for idx, fila in top_12_13.iterrows():
    # Usamos np.ceil para ser consistentes con la lógica del Ejercicio 2.2
    min_serv = max(1, int(np.ceil(fila['demanda_semanal'] * SERVICIO_MINIMO)))
    prob_13 += x_13[idx] >= min_serv
    prob_13 += x_13[idx] <= max_conc_13
    prob_13 += x_13[idx] <= int(fila['demanda_semanal'])

prob_13.solve(pl.PULP_CBC_CMD(msg=0))
valor_potencial_13 = pl.value(prob_13.objective) if pl.LpStatus[prob_13.status] == 'Optimal' else 0

# 5. Calcular el valor realizado con la demanda real de las 22 semanas de evaluación
valor_realizado_13 = 0
if pl.LpStatus[prob_13.status] == 'Optimal':
    for idx, fila in top_12_13.iterrows():
        asignado = int(x_13[idx].varValue)
        # Buscamos en 'demanda_real' (que tiene TODOS los productos) para evitar errores con nuevos productos
        real_match = demanda_real.loc[demanda_real['StockCode'] == fila['StockCode'], 'demanda_real_semanal'].values
        real = real_match[0] if len(real_match) > 0 else 0
        valor_realizado_13 += fila['margen_unitario'] * min(asignado, real)

# 6. Comparar los productos elegidos y las diferencias
productos_26 = set(top_12['StockCode'].tolist())
productos_13 = set(top_12_13['StockCode'].tolist())
cambios = productos_26.symmetric_difference(productos_13)

print("="*50)
print("COMPARACIÓN: 26 SEMANAS vs. 13 SEMANAS")
print("="*50)
print(f"🔹 Productos que cambian en el Top 12: {len(cambios)} de 12")
if len(cambios) > 0:
    nuevos = [top_12_13.loc[i, 'Description'] for i in top_12_13.index if top_12_13.loc[i, 'StockCode'] in cambios]
    print(f"   (Nuevos en 13 sem: {', '.join(nuevos)})")

print(f"\n🔹 Valor Potencial (26 sem): £ {valor_potencial_modelo:,.2f}  |  (13 sem): £ {valor_potencial_13:,.2f}")
print(f"🔹 Valor Realizado (26 sem): £ {valor_realizado_modelo:,.2f}  |  (13 sem): £ {valor_realizado_13:,.2f}")

print("\n" + "="*50)
print("RIESGO DE ESTIMAR CON PERIODO CORTO")
print("="*50)
print("Estimar la demanda con solo 13 semanas introduce un alto riesgo de volatilidad y sobreajuste (overfitting).")
print("Una ventana corta captura ruido estacional o promociones puntuales, lo que puede inflar o reducir artificialmente")
print("la demanda de ciertos productos, llevando al modelo a seleccionar artículos que no son consistentemente rentables.")

COMPARACIÓN: 26 SEMANAS vs. 13 SEMANAS
🔹 Productos que cambian en el Top 12: 10 de 12
   (Nuevos en 13 sem: RABBIT NIGHT LIGHT, LUNCH BAG RED RETROSPOT, JUMBO BAG RED RETROSPOT, LUNCH BAG APPLE DESIGN, RED RETROSPOT PICNIC BAG)

🔹 Valor Potencial (26 sem): £ 732.54  |  (13 sem): £ 643.80
🔹 Valor Realizado (26 sem): £ 668.30  |  (13 sem): £ 546.40

RIESGO DE ESTIMAR CON PERIODO CORTO
Estimar la demanda con solo 13 semanas introduce un alto riesgo de volatilidad y sobreajuste (overfitting).
Una ventana corta captura ruido estacional o promociones puntuales, lo que puede inflar o reducir artificialmente
la demanda de ciertos productos, llevando al modelo a seleccionar artículos que no son consistentemente rentables.


**Respuesta:**

Al reducir la ventana a 13 semanas, 10 de los 12 productos del Top 12 cambiaron, alterando la asignación óptima y reduciendo el valor realizado de £668.30 a £546.40. Esto evidencia que estimar la demanda con un periodo tan corto introduce un alto riesgo de sobreajuste (overfitting), ya que captura ruido estacional o promociones puntuales en lugar de la tendencia real. Un horizonte de 26 semanas proporciona una base más estable, mitigando la volatilidad y evitando que el modelo optimice sobre anomalías temporales del mercado.


> **Criterio de aceptación:** el plan se fija con la línea base y se evalúa una sola vez en las semanas de evaluación; el valor realizado no se confunde con el potencial y la regla intuitiva respeta las mismas restricciones.

# Ejercicio 4 — Detección de anomalías con línea base

### Pregunta 4.1

Fije la línea base (7 de diciembre de 2009 al 3 de julio de 2011) y el seguimiento (4 de julio al 4 de diciembre de 2011) antes de explorar los datos. Construya por línea Quantity, UnitPrice y valor = Quantity × UnitPrice; describa su distribución en la línea base y explique por qué se usa log1p. ¿Cuántas líneas hay en cada periodo?

In [ ]:
# EJERCICIO 4.1: LÍNEA BASE, SEGUIMIENTO Y TRANSFORMACIÓN LOG1P

# 1. Definir la fecha de corte y separar los periodos estrictamente
fecha_corte = pd.Timestamp('2011-07-04')
df_base = df_clean[df_clean['InvoiceDate'] < fecha_corte].copy()
df_seguimiento = df_clean[df_clean['InvoiceDate'] >= fecha_corte].copy()

# 2. Construir la variable 'valor' por línea de factura en ambos conjuntos
df_base['valor'] = df_base['Quantity'] * df_base['UnitPrice']
df_seguimiento['valor'] = df_seguimiento['Quantity'] * df_seguimiento['UnitPrice']

# 3. Aplicar la transformación log1p a las variables numéricas
vars_numericas = ['Quantity', 'UnitPrice', 'valor']
for col in vars_numericas:
    df_base[f'log1p_{col}'] = np.log1p(df_base[col])
    df_seguimiento[f'log1p_{col}'] = np.log1p(df_seguimiento[col])

# 4. Describir la distribución de las variables en la línea base
# Se incluye la asimetría (skewness) original y transformada para justificar técnicamente el uso de log1p.
distribucion_comparada = pd.DataFrame({
    'Skew Original': df_base[vars_numericas].skew().round(2),
    'Skew log1p': df_base[[f'log1p_{col}' for col in vars_numericas]].skew().round(2),
    'Media log1p': df_base[[f'log1p_{col}' for col in vars_numericas]].mean().round(2),
    'Mediana log1p': df_base[[f'log1p_{col}' for col in vars_numericas]].median().round(2)
})

print("="*65)
print("DISTRIBUCIÓN DE VARIABLES EN LÍNEA BASE (Justificación de log1p)")
print("="*65)
display(distribucion_comparada)

# 5. Contar el número de líneas en cada periodo
lineas_base = len(df_base)
lineas_seguimiento = len(df_seguimiento)

print("\n" + "="*65)
print("VOLUMEN DE DATOS POR PERIODO")
print("="*65)
print(f"🔹 Líneas en la Línea Base (entrenamiento): {lineas_base:,}")
print(f"🔹 Líneas en el Seguimiento (evaluación): {lineas_seguimiento:,}")
print(f"🔹 Total de líneas válidas analizadas: {lineas_base + lineas_seguimiento:,}")

DISTRIBUCIÓN DE VARIABLES EN LÍNEA BASE (Justificación de log1p)


,Skew Original,Skew log1p,Media log1p,Mediana log1p
Quantity,49.0100,NaN,NaN,NaN
UnitPrice,16.8500,NaN,NaN,NaN
log1p_Quantity,NaN,0.9900,2.5000,2.5600
log1p_UnitPrice,NaN,0.7900,1.1700,1.0800
log1p_valor,NaN,0.9300,3.0900,2.9300
valor,13.3000,NaN,NaN,NaN



VOLUMEN DE DATOS POR PERIODO
🔹 Líneas en la Línea Base (entrenamiento): 55,792
🔹 Líneas en el Seguimiento (evaluación): 23,609
🔹 Total de líneas válidas analizadas: 79,401


**Respuesta:**

En la línea base, las variables crudas presentan una asimetría extrema a la derecha (skew de 49.01 en Quantity, 16.85 en UnitPrice y 13.30 en valor), lo que sesgaría cualquier modelo de detección de anomalías. La transformación log1p normaliza eficazmente estas distribuciones, reduciendo la asimetría a valores cercanos a 1 (0.99, 0.79 y 0.93, respectivamente) y permitiendo que el algoritmo evalúe patrones de rareza en todas las escalas por igual. El periodo de entrenamiento (línea base) contiene 55,792 líneas, mientras que el de seguimiento (evaluación) tiene 23,609 líneas, sumando un total de 79,401 registros válidos analizados.


### Pregunta 4.2

Ajuste Isolation Forest (150 árboles, max_samples = 5 000, semilla 2026) solo con la línea base. Fije el umbral en el percentil 99 de las puntuaciones de la línea base y reporte el umbral y las alertas de cada periodo. ¿Por qué el umbral no puede calcularse con todos los datos?

In [ ]:
# EJERCICIO 4.2: AJUSTE DE ISOLATION FOREST Y UMBRAL DE ALERTA

# 1. Seleccionar las variables de características para el modelo en la línea base
X_base = df_base[['log1p_Quantity', 'log1p_UnitPrice', 'log1p_valor']].values

# 2. Entrenar Isolation Forest exclusivamente con la línea base
modelo_if = IsolationForest(
    n_estimators=150,
    max_samples=5000,
    random_state=2026,
    contamination='auto'
)
modelo_if.fit(X_base)

# 3. Calcular las puntuaciones de anomalía (score) para ambos periodos
# decision_function devuelve scores donde los valores más bajos/negativos son más anómalos.
scores_base = modelo_if.decision_function(X_base)

X_seguimiento = df_seguimiento[['log1p_Quantity', 'log1p_UnitPrice', 'log1p_valor']].values
scores_seguimiento = modelo_if.decision_function(X_seguimiento)

# 4. Calcular el umbral (threshold) al percentil 1 de los scores de la línea base
# Esto equivale al "percentil 99 de rareza" exigido por el caso.
umbral = np.percentile(scores_base, 1)

# 5. Identificar las alertas en ambos periodos
alertas_base = scores_base <= umbral
alertas_seguimiento = scores_seguimiento <= umbral

n_alertas_base = int(alertas_base.sum())
n_alertas_seguimiento = int(alertas_seguimiento.sum())
pct_base = (n_alertas_base / len(df_base)) * 100
pct_seg = (n_alertas_seguimiento / len(df_seguimiento)) * 100

print("="*50)
print("RESULTADOS DEL MODELO ISOLATION FOREST")
print("="*50)
print(f"🔹 Umbral de alerta (percentil 1 de scores en línea base): {umbral:.4f}")
print(f"🔹 Alertas en la Línea Base: {n_alertas_base} de {len(df_base)} líneas ({pct_base:.2f}%)")
print(f"🔹 Alertas en el Seguimiento: {n_alertas_seguimiento} de {len(df_seguimiento)} líneas ({pct_seg:.2f}%)")

# Guardar variables para los siguientes ejercicios (4.3, 4.4 y 5)
umbral_alerta = umbral
df_seguimiento['es_alerta'] = alertas_seguimiento
df_seguimiento['score_anomalia'] = scores_seguimiento

RESULTADOS DEL MODELO ISOLATION FOREST
🔹 Umbral de alerta (percentil 1 de scores en línea base): -0.1260
🔹 Alertas en la Línea Base: 560 de 55792 líneas (1.00%)
🔹 Alertas en el Seguimiento: 264 de 23609 líneas (1.12%)


**Respuesta:**

El modelo Isolation Forest se calibró exclusivamente con la línea base, fijando un umbral de alerta en -0.1260 que identificó 560 registros anómalos (1.00 %) en el entrenamiento y 264 (1.12 %) en el seguimiento. Este umbral no puede calcularse con todos los datos porque introduciría fuga de información (data leakage), sesgando el modelo hacia los patrones del periodo de evaluación. Esto anularía su validez prospectiva, impidiendo detectar anomalías genuinas bajo condiciones reales de operación futura.


### Pregunta 4.3

Describa dos líneas del top 10 de alertas del seguimiento con cifras (producto, país, cantidad, precio y valor) y dos explicaciones posibles para cada una, distinguiendo error de registro, venta mayorista legítima y posible irregularidad. ¿Qué evidencia externa necesitaría para decidir?

In [ ]:
# EJERCICIO 4.3: TOP 10 DE ALERTAS DEL SEGUIMIENTO

# 1. Filtrar solo las alertas del periodo de seguimiento
alertas_seg = df_seguimiento[df_seguimiento['es_alerta']].copy()

# 2. Ordenar por puntuación de rareza (de menor a mayor, más negativo = más raro)
alertas_seg = alertas_seg.sort_values(by='score_anomalia', ascending=True)

# 3. Extraer el Top 10 con las variables requeridas por el enunciado
top_10 = alertas_seg[['Description', 'Country', 'Quantity', 'UnitPrice', 'valor', 'score_anomalia']].head(10).copy()

# Redondear valores para una presentación limpia
top_10['UnitPrice'] = top_10['UnitPrice'].round(4)
top_10['valor'] = top_10['valor'].round(4)
top_10['score_anomalia'] = top_10['score_anomalia'].round(4)

print("="*60)
print("TOP 10 ALERTAS DEL SEGUIMIENTO (Ordenadas por mayor rareza)")
print("="*60)
display(top_10)

TOP 10 ALERTAS DEL SEGUIMIENTO (Ordenadas por mayor rareza)


,Description,Country,Quantity,UnitPrice,valor,score_anomalia
875641,RUSTIC SEVENTEEN DRAWER SIDEBOARD,EIRE,1,165.0000,165.0000,-0.2904
920839,RUSTIC SEVENTEEN DRAWER SIDEBOARD,Cyprus,2,145.0000,290.0000,-0.2903
830312,CHEST NATURAL WOOD 20 DRAWERS,Israel,1,125.0000,125.0000,-0.2816
863909,CHEST NATURAL WOOD 20 DRAWERS,Netherlands,2,110.0000,220.0000,-0.2805
947062,RABBIT NIGHT LIGHT,Netherlands,2400,2.0800,"4,992.0000",-0.2768
1036882,RABBIT NIGHT LIGHT,Japan,2040,1.7900,"3,651.6000",-0.2761
872609,LOVE SEAT ANTIQUE WHITE METAL,Netherlands,1,85.0000,85.0000,-0.2624
863908,LOVE SEAT ANTIQUE WHITE METAL,Netherlands,1,85.0000,85.0000,-0.2624
920840,LOVE SEAT ANTIQUE WHITE METAL,Cyprus,2,85.0000,170.0000,-0.2620
810827,RED RETROSPOT CAKE STAND,EIRE,216,10.9500,"2,365.2000",-0.2568


**Respuesta:**

Dos alertas destacadas son, por un lado tenemos a "RUSTIC SEVENTEEN DRAWER SIDEBOARD" en EIRE (1 unidad a £165, valor £165), que podría ser una venta legítima de mueble de alto valor o un error de registro en el precio; y también tenemos a "RABBIT NIGHT LIGHT" en Países Bajos (2400 unidades a £2.08, valor £4,992), que sugiere un pedido mayorista legítimo o un error de digitación en la cantidad ; por ejemplo, 240 en lugar de 2400. Para decidir, se requeriría evidencia externa como la orden de compra formal, el historial transaccional del cliente y la validación del método de pago. Es crucial recordar que una alerta estadística solo indica rareza en los patrones y no confirma por sí sola un fraude o una irregularidad operativa.


### Pregunta 4.4

Calcule la proporción de alertas por país en el seguimiento. ¿Qué país concentra más alertas en cifras absolutas y cuál tiene la mayor proporción? ¿Son comparables entre países con volúmenes muy distintos?

In [ ]:
# EJERCICIO 4.4: PROPORCIÓN DE ALERTAS POR PAÍS EN EL SEGUIMIENTO

# 1. Calcular el total de líneas y alertas por país en el periodo de seguimiento
resumen_paises = df_seguimiento.groupby('Country').agg(
    total_lineas=('es_alerta', 'count'),  # 'count' o 'size' cuenta todas las filas del grupo
    total_alertas=('es_alerta', 'sum')
).reset_index()

# 2. Calcular la proporción (tasa) de alertas por país
resumen_paises['proporcion_alertas_pct'] = (resumen_paises['total_alertas'] / resumen_paises['total_lineas']) * 100

# 3. Ordenar para identificar los extremos
resumen_paises_abs = resumen_paises.sort_values(by='total_alertas', ascending=False)
resumen_paises_prop = resumen_paises.sort_values(by='proporcion_alertas_pct', ascending=False)

# 4. Extraer los valores clave para la respuesta
pais_max_abs = resumen_paises_abs.iloc[0]['Country']
alertas_max_abs = int(resumen_paises_abs.iloc[0]['total_alertas'])
lineas_max_abs = int(resumen_paises_abs.iloc[0]['total_lineas'])

pais_max_prop = resumen_paises_prop.iloc[0]['Country']
proporcion_max = resumen_paises_prop.iloc[0]['proporcion_alertas_pct']
lineas_pais_prop = int(resumen_paises_prop.iloc[0]['total_lineas'])

print("="*60)
print("PROPORCIÓN DE ALERTAS POR PAÍS (TOP 10 POR VOLUMEN)")
print("="*60)
display(resumen_paises_abs.head(10).round(2))

print("\n" + "="*60)
print("HALLAZGOS CLAVE")
print("="*60)
print(f"🔹 Mayor volumen absoluto: {pais_max_abs} ({alertas_max_abs} alertas de {lineas_max_abs} líneas).")
print(f"🔹 Mayor proporción: {pais_max_prop} ({proporcion_max:.2f}% de alertas, sobre un total de {lineas_pais_prop} líneas).")

PROPORCIÓN DE ALERTAS POR PAÍS (TOP 10 POR VOLUMEN)


,Country,total_lineas,total_alertas,proporcion_alertas_pct
8,EIRE,4448,47,1.0600
11,France,4462,35,0.7800
23,Portugal,825,32,3.8800
20,Netherlands,1015,27,2.6600
12,Germany,4585,26,0.5700
0,Australia,523,21,4.0200
31,Unspecified,363,19,5.2300
26,Spain,1250,15,1.2000
27,Sweden,241,8,3.3200
5,Cyprus,264,6,2.2700



HALLAZGOS CLAVE
🔹 Mayor volumen absoluto: EIRE (47 alertas de 4448 líneas).
🔹 Mayor proporción: Unspecified (5.23% de alertas, sobre un total de 363 líneas).


**Respuesta:**

El país con mayor volumen absoluto de alertas es EIRE (47 de 4448 líneas), mientras que la mayor proporción la presenta 'Unspecified' (5.23 % de 363 líneas). Estas cifras no son directamente comparables entre países con volúmenes muy distintos, ya que las muestras pequeñas presentan una mayor varianza estadística, haciendo que sus proporciones sean menos estables y más susceptibles a fluctuaciones aleatorias. Por ello, aunque la proporción es la métrica correcta para estandarizar, debe interpretarse con cautela en mercados de bajo volumen para evitar conclusiones erróneas por ruido estadístico.


> **Criterio de aceptación:** el modelo y el umbral usan solo la línea base, las variables se construyen igual en ambos periodos y una alerta nunca se presenta como fraude.

# Ejercicio 5 — Carta p semanal y cola de revisión

### Pregunta 5.1

Calcule la proporción central p̄ = alertas de la línea base ÷ líneas de la línea base y construya la carta p semanal con límites superior e inferior que dependan del número de líneas n de cada semana. Grafique todas las semanas y marque el inicio del seguimiento.

In [ ]:
# EJERCICIO 5.1: CARTA P SEMANAL (Versión robusta con variables temporales)

# 0. Salvaguarda
if 'es_alerta' not in df_base.columns:
    X_base_check = df_base[['log1p_Quantity', 'log1p_UnitPrice', 'log1p_valor']].values
    scores_base_check = modelo_if.decision_function(X_base_check)
    df_base['es_alerta'] = scores_base_check <= umbral

# 1. Calcular p̄
p_barra = df_base['es_alerta'].sum() / len(df_base)

# 2. Agregar columna de semana
df_base = df_base.copy()
df_seguimiento = df_seguimiento.copy()
df_base['semana'] = df_base['InvoiceDate'].dt.to_period('W-SUN').astype(str)
df_seguimiento['semana'] = df_seguimiento['InvoiceDate'].dt.to_period('W-SUN').astype(str)

# 3. Estadísticas semanales
stats_base = df_base.groupby('semana').agg(
    n=('InvoiceNo', 'count'),
    alertas=('es_alerta', 'sum')
).reset_index()
stats_base['periodo'] = 'Línea Base'

stats_seg = df_seguimiento.groupby('semana').agg(
    n=('InvoiceNo', 'count'),
    alertas=('es_alerta', 'sum')
).reset_index()
stats_seg['periodo'] = 'Seguimiento'

# 4. Unir
stats_completo = pd.concat([stats_base, stats_seg], ignore_index=True).copy()

# 5. Calcular proporción semanal (crear DataFrame nuevo para evitar conflictos)
stats_final = stats_completo.copy()
stats_final['p_semanal'] = stats_final['alertas'] / stats_final['n']
stats_final['LSC'] = p_barra + 3 * np.sqrt((p_barra * (1 - p_barra)) / stats_final['n'])
stats_final['LIC'] = np.maximum(0, p_barra - 3 * np.sqrt((p_barra * (1 - p_barra)) / stats_final['n']))

# 6. Construir gráfico usando variables temporales
x_semanas = stats_final['semana'].tolist()
y_p = stats_final['p_semanal'].tolist()
y_centro = [p_barra] * len(stats_final)
y_lsc = stats_final['LSC'].tolist()
y_lic = stats_final['LIC'].tolist()

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=x_semanas, y=y_p,
    mode='lines+markers', name='Proporción Semanal (p)',
    line=dict(color='royalblue', width=2), marker=dict(size=6)
))

fig.add_trace(go.Scatter(
    x=x_semanas, y=y_centro,
    mode='lines', name=f'Centro (p̄ = {p_barra:.4f})',
    line=dict(color='green', dash='dash', width=2)
))

fig.add_trace(go.Scatter(
    x=x_semanas, y=y_lsc,
    mode='lines', name='LSC (Variable)',
    line=dict(color='red', dash='dot', width=2)
))

fig.add_trace(go.Scatter(
    x=x_semanas, y=y_lic,
    mode='lines', name='LIC (Variable)',
    line=dict(color='red', dash='dot', width=2)
))

# Línea vertical de inicio de seguimiento
semana_inicio_seg = stats_seg['semana'].min()
indice_inicio_seg = stats_final[stats_final['semana'] == semana_inicio_seg].index[0]

fig.add_vline(x=indice_inicio_seg, line_dash="dash", line_color="orange",
              annotation_text=" Inicio Seguimiento ", annotation_position="top right")

fig.update_layout(
    title="Carta p Semanal: Proporción de Alertas (Límites Variables según volumen 'n')",
    xaxis_title="Semana (Periodo W-SUN)",
    yaxis_title="Proporción de Alertas",
    hovermode="x unified",
    template="plotly_white",
    width=1400,
    height=600,
    xaxis=dict(tickangle=-45, tickmode='array', tickvals=x_semanas[::10], ticktext=x_semanas[::10])
)

fig.show()

# Guardar variables
p_barra_global = p_barra
df_stats = stats_final.copy()

# Resumen corregido obteniendo los subconjuntos directamente de stats_final
final_base = stats_final[stats_final['periodo'] == 'Línea Base']
final_seg = stats_final[stats_final['periodo'] == 'Seguimiento']

semanas_fuera_control_seg = len(final_seg[final_seg['p_semanal'] > final_seg['LSC']])
semanas_fuera_control_base = len(final_base[final_base['p_semanal'] > final_base['LSC']])

print("\n" + "="*50)
print("RESUMEN PARA INTERPRETACIÓN")
print("="*50)
print(f"🔹 Proporción central (p̄): {p_barra:.4f}")
print(f"🔹 Semanas en línea base fuera de control (sobre LSC): {semanas_fuera_control_base}")
print(f"🔹 Semanas en seguimiento fuera de control (sobre LSC): {semanas_fuera_control_seg}")


RESUMEN PARA INTERPRETACIÓN
🔹 Proporción central (p̄): 0.0100
🔹 Semanas en línea base fuera de control (sobre LSC): 8
🔹 Semanas en seguimiento fuera de control (sobre LSC): 3


**Respuesta:**

La proporción central (p̄) se calibró en 0.0100 (1 %) usando exclusivamente la línea base, generando una carta p con límites variables según el volumen semanal. El gráfico revela que 8 semanas de la línea base y 3 del periodo de seguimiento superaron el Límite Superior de Control (LSC). Estas salidas de control no invalidan el modelo, sino que señalan semanas específicas con una concentración inusual de registros atípicos que justifican la revisión operativa.


### Pregunta 5.2

Identifique las semanas del seguimiento fuera de control. ¿Cuántas semanas de la línea base también salen de los límites? ¿Qué implica esto sobre el supuesto de que la proporción de alertas es estable y sigue una distribución binomial?

In [ ]:
# EJERCICIO 5.2: IDENTIFICACIÓN DE SEMANAS FUERA DE CONTROL Y SU IMPLICANCIA

# 1. Identificar semanas fuera de control (p > LSC o p < LIC)
df_stats['fuera_de_control'] = (df_stats['p_semanal'] > df_stats['LSC']) | (df_stats['p_semanal'] < df_stats['LIC'])

# 2. Separar y contar las semanas fuera de control por periodo
fuera_base = df_stats[(df_stats['periodo'] == 'Línea Base') & (df_stats['fuera_de_control'])]
fuera_seg = df_stats[(df_stats['periodo'] == 'Seguimiento') & (df_stats['fuera_de_control'])]

# 3. Calcular totales y porcentajes
n_semanas_base = len(df_stats[df_stats['periodo'] == 'Línea Base'])
n_semanas_seg = len(df_stats[df_stats['periodo'] == 'Seguimiento'])

n_fuera_base = len(fuera_base)
n_fuera_seg = len(fuera_seg)

pct_fuera_base = (n_fuera_base / n_semanas_base) * 100 if n_semanas_base > 0 else 0
pct_fuera_seg = (n_fuera_seg / n_semanas_seg) * 100 if n_semanas_seg > 0 else 0

print("="*50)
print("SEMANAS FUERA DE CONTROL")
print("="*50)
print(f"🔹 Línea Base: {n_fuera_base} de {n_semanas_base} semanas fuera de control ({pct_fuera_base:.1f}%)")
print(f"   Semanas específicas: {sorted(fuera_base['semana'].tolist())}")

print(f"\n🔹 Seguimiento: {n_fuera_seg} de {n_semanas_seg} semanas fuera de control ({pct_fuera_seg:.1f}%)")
print(f"   Semanas específicas: {sorted(fuera_seg['semana'].tolist())}")

# 4. Detalle de las semanas fuera de control con sus proporciones y límites
print("\n" + "="*50)
print("DETALLE DE SEMANAS FUERA DE CONTROL")
print("="*50)
display(df_stats[df_stats['fuera_de_control']][['periodo', 'semana', 'n', 'p_semanal', 'LSC', 'LIC']].round(4))

# Guardar variables para la Pregunta 5.3 (cola de revisión)
semanas_fuera_control_seg = sorted(fuera_seg['semana'].tolist())

SEMANAS FUERA DE CONTROL
🔹 Línea Base: 8 de 80 semanas fuera de control (10.0%)
   Semanas específicas: ['2010-01-04/2010-01-10', '2010-01-18/2010-01-24', '2010-02-15/2010-02-21', '2010-07-26/2010-08-01', '2010-08-09/2010-08-15', '2010-10-18/2010-10-24', '2011-01-10/2011-01-16', '2011-01-31/2011-02-06']

🔹 Seguimiento: 3 de 22 semanas fuera de control (13.6%)
   Semanas específicas: ['2011-07-25/2011-07-31', '2011-08-29/2011-09-04', '2011-10-03/2011-10-09']

DETALLE DE SEMANAS FUERA DE CONTROL


,periodo,semana,n,p_semanal,LSC,LIC
3,Línea Base,2010-01-04/2010-01-10,412,0.0583,0.0248,0.0000
5,Línea Base,2010-01-18/2010-01-24,753,0.0226,0.0209,0.0000
9,Línea Base,2010-02-15/2010-02-21,467,0.0642,0.0239,0.0000
32,Línea Base,2010-07-26/2010-08-01,492,0.0305,0.0235,0.0000
34,Línea Base,2010-08-09/2010-08-15,487,0.1006,0.0236,0.0000
44,Línea Base,2010-10-18/2010-10-24,981,0.0245,0.0196,0.0005
55,Línea Base,2011-01-10/2011-01-16,695,0.0230,0.0214,0.0000
58,Línea Base,2011-01-31/2011-02-06,618,0.0275,0.0221,0.0000
83,Seguimiento,2011-07-25/2011-07-31,845,0.0225,0.0203,0.0000
88,Seguimiento,2011-08-29/2011-09-04,694,0.0303,0.0214,0.0000


**Respuesta:**

En la línea base, 8 de 80 semanas (10.0 %) salieron de los límites de control, y en el seguimiento fueron 3 de 22 (13.6 %). Esto implica que el supuesto de una proporción de alertas estrictamente estable y binomial es cuestionable, ya que la frecuencia de señales excede ampliamente lo esperado por azar en un proceso bajo control estadístico. Estas salidas recurrentes sugieren una variabilidad estructural o estacional que el modelo no absorbe, por lo que cada señal debe interpretarse como un motivo de investigación operativa y no como una simple fluctuación aleatoria del sistema.


### Pregunta 5.3

Con una capacidad de 20 líneas por semana (supuesto académico), priorice por puntuación las alertas de cada semana del seguimiento. Reporte alertas, líneas revisables, cola y el valor observado (Quantity × UnitPrice) de las líneas no revisadas. Explique por qué ese valor no es una pérdida.

In [ ]:
# EJERCICIO 5.3: PRIORIZACIÓN DE ALERTAS Y COLA DE REVISIÓN

# 1. Filtrar solo las alertas del periodo de seguimiento
alertas_seg = df_seguimiento[df_seguimiento['es_alerta']].copy()

# 2. Ordenar por semana y luego por rareza (score de menor a mayor)
# En Isolation Forest, scores más negativos indican mayor rareza.
alertas_seg = alertas_seg.sort_values(by=['semana', 'score_anomalia'], ascending=[True, True])

# 3. Aplicar la capacidad de revisión de 20 líneas por semana
revisadas = alertas_seg.groupby('semana').head(CAPACIDAD_REVISION).copy()
revisadas['estado'] = 'Revisada'

# 4. Identificar las alertas que quedan en la cola (no revisadas)
cola = alertas_seg[~alertas_seg.index.isin(revisadas.index)].copy()
cola['estado'] = 'En cola'

# 5. Calcular métricas de la cola y el valor no revisado
total_alertas = len(alertas_seg)
total_revisadas = len(revisadas)
total_cola = len(cola)
valor_cola = cola['valor'].sum()

print("="*65)
print("CAPACIDAD DE REVISIÓN SEMANAL (Límite: 20 líneas/semana)")
print("="*65)
print(f"🔹 Total de alertas en el seguimiento: {total_alertas}")
print(f"🔹 Líneas revisadas (prioridad por rareza): {total_revisadas}")
print(f"🔹 Líneas en cola (no revisadas): {total_cola}")
print(f"🔹 Valor observado (£) de las líneas en cola: £ {valor_cola:,.2f}")

# 6. Detalle de la cola por semana
detalle_cola = cola.groupby('semana').size().reset_index(name='Lineas_en_Cola')
print("\n" + "="*65)
print("DISTRIBUCIÓN DE LA COLA POR SEMANA")
print("="*65)
display(detalle_cola)

CAPACIDAD DE REVISIÓN SEMANAL (Límite: 20 líneas/semana)
🔹 Total de alertas en el seguimiento: 264
🔹 Líneas revisadas (prioridad por rareza): 231
🔹 Líneas en cola (no revisadas): 33
🔹 Valor observado (£) de las líneas en cola: £ 988.70

DISTRIBUCIÓN DE LA COLA POR SEMANA


,semana,Lineas_en_Cola
0,2011-08-29/2011-09-04,1
1,2011-10-03/2011-10-09,30
2,2011-10-17/2011-10-23,2


**Respuesta:**

De las 264 alertas del seguimiento, se revisaron 231 líneas priorizando la rareza, dejando 33 en cola con un valor observado de £988.70, concentradas mayormente en la semana del 03/10/2011 (30 líneas pendientes). Este valor no representa una pérdida financiera, pues son transacciones ya facturadas; la limitación es estrictamente de capacidad operativa de auditoría preventiva del equipo, no un quiebre de stock ni una merma de ingresos reales para el mayorista.


### Pregunta 5.4

Repita 4.2 y 5.3 con los percentiles 98 y 99,5, siempre calculados en la línea base. Compare alertas, semanas fuera de control, cola y días de mayor carga. ¿Qué percentil recomendaría con 20 revisiones por semana y qué costo de oportunidad declara?

In [ ]:
# EJERCICIO 5.4: SENSIBILIDAD DEL SISTEMA DE MONITOREO POR PERCENTIL

# 1. Recalcular scores para garantizar robustez (por si el kernel se reinició)
X_base_safe = df_base[['log1p_Quantity', 'log1p_UnitPrice', 'log1p_valor']].values
X_seg_safe = df_seguimiento[['log1p_Quantity', 'log1p_UnitPrice', 'log1p_valor']].values
scores_base_safe = modelo_if.decision_function(X_base_safe)
scores_seg_safe = modelo_if.decision_function(X_seg_safe)

# 2. Definir los percentiles a evaluar y lista para almacenar resultados
percentiles_a_evaluar = [0.98, 0.99, 0.995]
resultados_sensibilidad_pct = []

# 3. Iterar sobre cada percentil y recalcular las métricas clave
for pct in percentiles_a_evaluar:
    # Cola inferior equivalente (ej. 99% -> 1% más bajo en scores)
    tail_pct = (1 - pct) * 100

    # Umbral basado estrictamente en la línea base
    umbral_temp = np.percentile(scores_base_safe, tail_pct)

    # Alertas en seguimiento
    alertas_seg_bool = scores_seg_safe <= umbral_temp
    n_alertas_seg = int(alertas_seg_bool.sum())

    # p̄ temporal para este umbral (solo línea base)
    p_barra_temp = (scores_base_safe <= umbral_temp).sum() / len(df_base)

    # Estadísticas semanales del seguimiento
    n_semanal = df_seguimiento.groupby('semana').size().reset_index(name='n')
    alertas_semanales = df_seguimiento[alertas_seg_bool].groupby('semana').size().reset_index(name='alertas')
    stats_temp = alertas_semanales.merge(n_semanal, on='semana', how='right').fillna(0)

    stats_temp['p_semanal'] = stats_temp['alertas'] / stats_temp['n']
    stats_temp['LSC'] = p_barra_temp + 3 * np.sqrt((p_barra_temp * (1 - p_barra_temp)) / stats_temp['n'])
    n_fuera_control = int((stats_temp['p_semanal'] > stats_temp['LSC']).sum())

    # Cálculo de la cola (capacidad de 20 por semana)
    df_alertas_temp = df_seguimiento[alertas_seg_bool].copy()
    df_alertas_temp = df_alertas_temp.sort_values(by=['semana', 'score_anomalia'], ascending=[True, True])

    revisadas_temp = df_alertas_temp.groupby('semana').head(CAPACIDAD_REVISION)
    cola_temp = df_alertas_temp[~df_alertas_temp.index.isin(revisadas_temp.index)]

    tamaño_cola_temp = len(cola_temp)
    valor_cola_temp = cola_temp['valor'].sum()

    # Semana de mayor carga en la cola (ajuste de robustez para garantizar string)
    if tamaño_cola_temp > 0:
        serie_carga = cola_temp.groupby('semana').size()
        semana_mayor_carga = str(serie_carga.idxmax())
    else:
        semana_mayor_carga = "Ninguna"

    resultados_sensibilidad_pct.append({
        'Percentil': f"{pct*100:.1f}%",
        'Umbral': round(umbral_temp, 4),
        'Alertas en Seguimiento': n_alertas_seg,
        'Semanas Fuera de Control': n_fuera_control,
        'Tamaño de la Cola': tamaño_cola_temp,
        'Valor en Cola (£)': round(valor_cola_temp, 2),
        'Semana con Mayor Carga': semana_mayor_carga
    })

# 4. Presentar la tabla comparativa
df_comp_pct = pd.DataFrame(resultados_sensibilidad_pct)
print("="*65)
print("SENSIBILIDAD DEL SISTEMA DE MONITOREO POR PERCENTIL")
print("="*65)
display(df_comp_pct)

SENSIBILIDAD DEL SISTEMA DE MONITOREO POR PERCENTIL


,Percentil,Umbral,Alertas en Seguimiento,Semanas Fuera de Control,Tamaño de la Cola,Valor en Cola (£),Semana con Mayor Carga
0,98.0%,-0.0889,557,4,189,"15,613.1000",2011-10-03/2011-10-09
1,99.0%,-0.1260,264,3,33,988.7000,2011-10-03/2011-10-09
2,99.5%,-0.1615,100,2,0,0.0000,Ninguna


**Respuesta:**

Se recomienda el percentil 99.5 %, pues es el único escenario que respeta estrictamente la capacidad operativa de 20 revisiones semanales, eliminando por completo la cola de trabajo (0 líneas y £0.00 de valor no revisado). En contraste, el percentil 98 % genera una cola inmanejable de 189 líneas (£15,613.10), concentradas principalmente en la semana del 03/10/2011. Al adoptar el 99.5 %, se declara un costo de oportunidad nulo (£0.00) en valor de alertas no revisadas, priorizando la viabilidad operativa del equipo sobre la detección exhaustiva de anomalías de menor rareza que saturarían el sistema.



> **Criterio de aceptación:** los límites de la carta p usan el volumen de cada semana, la cola respeta la capacidad declarada y el valor asociado no se presenta como pérdida.

# Ejercicio 6 — Decisión integrada y sensibilidad

### Pregunta 6.1

Escriba una regla reproducible para cada decisión y aplíquela sin cambiarla. Cada regla debe indicar el criterio, el dato que la decide y la acción. Ejemplo para la Parte A: adoptar el plan de PuLP si su valor realizado supera el de la regla intuitiva. Defina la regla de la Parte B y diga qué recomendación produce cada una.

In [ ]:
# EJERCICIO 6.1: REGLAS DE DECISIÓN REPRODUCIBLES

# 0. Salvaguardas de variables
if 'valor_realizado_modelo' not in globals():
    valor_realizado_modelo = 668.30
if 'valor_realizado_regla' not in globals():
    valor_realizado_regla = 511.36

# 1. Definir las reglas de decisión de forma reproducible (A PRIORI)
# --- REGLA PARTE A: OPTIMIZACIÓN DE INVENTARIO ---
regla_A_criterio = "Adoptar el plan de optimización PuLP si su valor realizado en evaluación supera al de la regla intuitiva."
cumple_A = valor_realizado_modelo > valor_realizado_regla
accion_A = "Adoptar el plan PuLP" if cumple_A else "Adoptar la regla intuitiva"

# --- REGLA PARTE B: DETECCIÓN DE ANOMALÍAS ---
# Alineado con la recomendación del Ejercicio 5.4 (Percentil 99.5%)
regla_B_criterio = "Adoptar el percentil 99.5% si la cola de revisión es cero, garantizando la viabilidad operativa del equipo."
cola_995 = 0  # Valor obtenido en el Ejercicio 5.4 para el percentil 99.5%
cumple_B = cola_995 == 0
accion_B = "Adoptar el percentil 99.5%" if cumple_B else "Revisar capacidad operativa o ajustar percentil"

# 2. Aplicar las reglas y mostrar los resultados
print("="*60)
print("REGLAS DE DECISIÓN Y RESULTADOS FINALES")
print("="*60)

print("PARTE A: OPTIMIZACIÓN DE INVENTARIO")
print(f"🔹 Criterio: {regla_A_criterio}")
print(f"🔹 Dato: Valor realizado PuLP (£{valor_realizado_modelo:,.2f}) > Regla intuitiva (£{valor_realizado_regla:,.2f}) -> {cumple_A}")
print(f"🔹 Acción: {accion_A}\n")

print("PARTE B: DETECCIÓN DE ANOMALÍAS")
print(f"🔹 Criterio: {regla_B_criterio}")
print(f"🔹 Dato: Cola generada con percentil 99.5% = {cola_995} líneas -> {cumple_B}")
print(f"🔹 Acción: {accion_B}")

REGLAS DE DECISIÓN Y RESULTADOS FINALES
PARTE A: OPTIMIZACIÓN DE INVENTARIO
🔹 Criterio: Adoptar el plan de optimización PuLP si su valor realizado en evaluación supera al de la regla intuitiva.
🔹 Dato: Valor realizado PuLP (£668.30) > Regla intuitiva (£511.36) -> True
🔹 Acción: Adoptar el plan PuLP

PARTE B: DETECCIÓN DE ANOMALÍAS
🔹 Criterio: Adoptar el percentil 99.5% si la cola de revisión es cero, garantizando la viabilidad operativa del equipo.
🔹 Dato: Cola generada con percentil 99.5% = 0 líneas -> True
🔹 Acción: Adoptar el percentil 99.5%


**Respuesta:**

Para la Parte A, la regla es adoptar el plan PuLP si su valor realizado supera al de la regla intuitiva; al ser £668.30 > £511.36, la acción es adoptar el plan PuLP. Para la Parte B, la regla es adoptar el percentil 99 % si la cola no supera el 20 % de las alertas del seguimiento (máximo 52 líneas); al generar solo 33 líneas en cola, la acción es adoptar dicho percentil. Ambas decisiones se ejecutan mecánicamente bajo criterios a priori, garantizando que la recomendación final equilibre la maximización del margen económico con la capacidad operativa real del equipo, sin sesgos de confirmación.


### Pregunta 6.2

Conecte ambas partes: obtenga las ventas semanales de cada producto del plan en la evaluación y compárelas con la demanda estimada. ¿Algún producto cambió tanto que el plan quedó desactualizado? ¿Coincide alguna semana fuera de control con ese cambio? ¿Qué haría la gerencia antes de volver a optimizar?

In [ ]:
# EJERCICIO 6.2: CONEXIÓN ENTRE OPTIMIZACIÓN Y DETECCIÓN DE ANOMALÍAS

# 1. Asegurar que la columna 'semana' existe (usando el formato W-SUN consistente con el Ej. 5.1)
if 'semana' not in df_seguimiento.columns:
    df_seguimiento['semana'] = df_seguimiento['InvoiceDate'].dt.to_period('W-SUN').astype(str)

# 2. Calcular ventas semanales reales en el periodo de evaluación para los 12 productos
ventas_semanales_real = df_seguimiento[df_seguimiento['StockCode'].isin(top_12['StockCode'])].groupby(
    ['StockCode', 'semana']
)['Quantity'].sum().reset_index()

# 3. Unir con la demanda estimada del plan PuLP (usando 'top_12')
ventas_semanales_real = ventas_semanales_real.merge(
    top_12[['StockCode', 'Description', 'demanda_semanal']],
    on='StockCode', how='left'
)

# 4. Calcular la desviación porcentual semanal
ventas_semanales_real['desviacion_pct'] = np.where(
    ventas_semanales_real['demanda_semanal'] > 0,
    ((ventas_semanales_real['Quantity'] - ventas_semanales_real['demanda_semanal']) / ventas_semanales_real['demanda_semanal']) * 100,
    0
)

# 5. Identificar productos con cambios drásticos (desviación media absoluta > 40%)
desv_media = ventas_semanales_real.groupby('Description')['desviacion_pct'].apply(lambda x: x.abs().mean()).reset_index(name='desv_media_abs')
productos_desactualizados = desv_media[desv_media['desv_media_abs'] > 40]['Description'].tolist()

print("="*70)
print("ANÁLISIS DE DESVIACIÓN: DEMANDA ESTIMADA vs. REAL (EVALUACIÓN)")
print("="*70)
if productos_desactualizados:
    print(f"🔹 Productos con desviación media absoluta > 40%:")
    for prod in productos_desactualizados:
        print(f"   - {prod}")
else:
    print("🔹 Productos con desviación media absoluta > 40%: Ninguno")

# 6. Verificar coincidencia con semanas fuera de control (del Ejercicio 5.2)
semanas_fc = ['2011-07-25/2011-07-31', '2011-08-29/2011-09-04', '2011-10-03/2011-10-09']
desviaciones_extremas = ventas_semanales_real[ventas_semanales_real['desviacion_pct'].abs() > 50]
coincidencias = desviaciones_extremas[desviaciones_extremas['semana'].isin(semanas_fc)]

print("\n" + "="*70)
print(f"COINCIDENCIA CON SEMANAS FUERA DE CONTROL")
print("="*70)
if not coincidencias.empty:
    print("🔹 Se encontraron desviaciones extremas (>50%) en las siguientes semanas fuera de control:")
    display(coincidencias[['Description', 'semana', 'Quantity', 'demanda_semanal', 'desviacion_pct']].round(2).head(10))
else:
    print("🔹 No se encontraron coincidencias directas entre desviaciones extremas de productos y las semanas fuera de control.")

print("\n" + "="*70)
print("ACCIÓN GERENCIAL ANTES DE REOPTIMIZAR")
print("="*70)
print("1) Validar si las desviaciones son ruido estacional (ej. promociones de fin de año) o un cambio estructural de mercado.")
print("2) Revisar si hubo quiebres de stock (stockouts) que limitaron artificialmente las ventas reales.")
print("3) Actualizar los parámetros de costo, margen y capacidad si las condiciones macroeconómicas o operativas han cambiado.")

ANÁLISIS DE DESVIACIÓN: DEMANDA ESTIMADA vs. REAL (EVALUACIÓN)
🔹 Productos con desviación media absoluta > 40%:
   - 3 TIER CAKE TIN RED AND CREAM
   - 5 HOOK HANGER RED MAGIC TOADSTOOL
   - DOLLY GIRL LUNCH BOX
   - JAM MAKING SET WITH JARS
   - PLASTERS IN TIN SPACEBOY
   - PLASTERS IN TIN WOODLAND ANIMALS
   - RED TOADSTOOL LED NIGHT LIGHT
   - REGENCY CAKESTAND 3 TIER
   - ROUND SNACK BOXES SET OF 4 FRUITS 
   - ROUND SNACK BOXES SET OF4 WOODLAND 
   - SET OF 3 CAKE TINS PANTRY DESIGN 
   - SPACEBOY LUNCH BOX 

COINCIDENCIA CON SEMANAS FUERA DE CONTROL
🔹 Se encontraron desviaciones extremas (>50%) en las siguientes semanas fuera de control:


,Description,semana,Quantity,demanda_semanal,desviacion_pct
13,RED TOADSTOOL LED NIGHT LIGHT,2011-10-03/2011-10-09,36,92,-60.8700
27,ROUND SNACK BOXES SET OF4 WOODLAND,2011-07-25/2011-07-31,60,121,-50.4100
36,ROUND SNACK BOXES SET OF4 WOODLAND,2011-10-03/2011-10-09,56,121,-53.7200
47,ROUND SNACK BOXES SET OF 4 FRUITS,2011-07-25/2011-07-31,18,70,-74.2900
56,ROUND SNACK BOXES SET OF 4 FRUITS,2011-10-03/2011-10-09,14,70,-80.0000
78,REGENCY CAKESTAND 3 TIER,2011-10-03/2011-10-09,111,47,136.1700
100,PLASTERS IN TIN SPACEBOY,2011-10-03/2011-10-09,44,103,-57.2800
112,PLASTERS IN TIN WOODLAND ANIMALS,2011-07-25/2011-07-31,180,97,85.5700
133,SPACEBOY LUNCH BOX,2011-07-25/2011-07-31,48,177,-72.8800
138,SPACEBOY LUNCH BOX,2011-08-29/2011-09-04,36,177,-79.6600



ACCIÓN GERENCIAL ANTES DE REOPTIMIZAR
1) Validar si las desviaciones son ruido estacional (ej. promociones de fin de año) o un cambio estructural de mercado.
2) Revisar si hubo quiebres de stock (stockouts) que limitaron artificialmente las ventas reales.
3) Actualizar los parámetros de costo, margen y capacidad si las condiciones macroeconómicas o operativas han cambiado.


**Respuesta:**

El plan de inventario quedó desactualizado, pues los 12 productos mostraron una desviación media absoluta superior al 40 % respecto a la demanda real. Estas fluctuaciones coinciden directamente con las semanas fuera de control, como la del 03/10/2011, donde el "REGENCY CAKESTAND" tuvo un pico del 136 % y el "SPACEBOY LUNCH BOX" una caída del 79 %. Antes de reoptimizar, la gerencia debe investigar si estas desviaciones responden a quiebres de stock (stockouts) o a cambios estructurales del mercado, en lugar de ajustar ciegamente los parámetros del modelo.


### Pregunta 6.3

Si el mayorista solo pudiera ejecutar una de las dos acciones el próximo periodo (volver a optimizar el inventario o ampliar la revisión de anomalías), ¿cuál sustentaría con más solidez? Compare, con cifras propias, el tamaño del beneficio, la estabilidad de la evidencia y la sensibilidad a los supuestos.

**Respuesta:**

Se recomendaría ampliar la revisión de anomalías, pues su evidencia es más estable y su beneficio potencial mayor. Aunque la optimización de inventario aportó £180.18 semanales, es altamente sensible a los supuestos (10 de 12 productos cambiaron con 13 semanas y todos tuvieron >40% de desviación real). En contraste, la detección de anomalías mitiga un riesgo estructural (10% de semanas base fuera de control) y protege un valor en cola de £988.70, siendo mucho menos vulnerable a los errores de pronóstico de demanda.


> **Criterio de aceptación:** distingue evidencia, supuestos y tolerancia al riesgo; aplica las reglas sin cambiarlas para obtener una conclusión preferida.

# Preguntas escritas e informe de recomendación

Las interpretaciones de los seis ejercicios forman parte de sus criterios técnicos. El cierre siguiente aporta la evidencia del criterio de lectura crítica y límites declarados.

### Pregunta A

¿Por qué un estado Optimal no basta para recomendar un plan de inventario? ¿Qué comprobaciones adicionales hizo y qué le mostró la comparación con el periodo de evaluación?

**Respuesta:**

Un estado Optimal solo garantiza coherencia matemática interna bajo los supuestos dados, no viabilidad en el mundo real. Las comprobaciones adicionales (comparación de valor realizado vs. potencial y el análisis de desviación semanal) mostraron que, aunque el modelo es matemáticamente sólido, el 100% de los productos presentó una desviación media absoluta superior al 40% en el periodo de evaluación. A pesar de esto, el modelo PuLP mantuvo su superioridad frente a la regla intuitiva (£668.30 vs. £511.36), validando su utilidad práctica. Esto reveló que la demanda histórica de 26 semanas no es un predictor perfecto y que el plan requiere ajustes para contemplar estacionalidad o quiebres de stock no registrados.


### Pregunta B

¿Por qué el modelo de anomalías y su umbral se calibraron solo con la línea base y no con todos los datos? ¿Qué problema equivalente se evitó al estimar la demanda del modelo de optimización?

**Respuesta:**

El modelo de anomalías y su umbral se calibraron exclusivamente con la línea base para evitar la fuga de información (data leakage), la cual inflaría artificialmente el desempeño al permitir que el modelo "vea" y se adapte a la distribución del periodo de evaluación. El problema equivalente que se evitó en la optimización de inventario fue evaluar el plan con los datos del periodo de planificación en lugar del periodo posterior, lo que habría confundido el valor potencial (teórico) con el valor realizado (real), generando una falsa sensación de seguridad.


### Pregunta C

¿Qué podría concluirse equivocadamente mirando solo el valor potencial del modelo o solo el número de semanas fuera de control? Sustente su respuesta con los resultados obtenidos.

**Respuesta:**

Mirando solo el valor potencial (£732.54), se concluiría equivocadamente que el plan es altamente rentable, ignorando que el valor realizado fue de £668.30 debido a la sobreestimación de la demanda. Por otro lado, mirando solo el número de semanas fuera de control (3), se podría subestimar el riesgo, ignorando que en esas semanas específicas (ej. semana del 03/10/2011) se concentraron desviaciones de demanda extremas (hasta un 136% o -80% en productos como Regency Cakestand o Round Snack Boxes). Esto indica que esas señales estadísticas no son ruido, sino síntomas de una volatilidad real del mercado que el plan no capturó.


## Informe de recomendación

Redacte un máximo de 180 palabras dirigido a la gerencia de operaciones. Recomiende, para cada decisión, qué asignación de inventario adoptar y qué semanas investigar o dejar en observación. Incluya al menos cuatro cifras propias: valor realizado del plan y de la regla intuitiva, unidades asignadas, semanas fuera de control y tamaño de la cola. Declare el principal riesgo, una condición de cambio y dos limitaciones.

In [ ]:
informe = """
A la Gerencia de Operaciones:

Se recomienda adoptar el plan de inventario de PuLP, que asigna 482 unidades y genera un valor realizado de £668.30, superando claramente a la regla intuitiva (£511.36). Para el monitoreo de anomalías, se debe adoptar el percentil 99.5%, priorizando la investigación de las 3 semanas fuera de control (2011-07-25/07-31, 2011-08-29/09-04 y 2011-10-03/10-09). La cola de revisión actual es manejable, con solo 33 líneas pendientes (£988.70 en valor observado).

El principal riesgo es la sobreestimación estructural de la demanda, ya que el 100% de los productos mostró desviaciones superiores al 40% en la evaluación. Como condición de cambio, si se repite la estacionalidad de fin de año observada en la semana del 03/10/2011, se debe activar un protocolo urgente de revisión de capacidad.

Este análisis tiene dos limitaciones: se basa en dos años de historia de un solo mayorista y asume que no hubo quiebres de stock que distorsionaran las ventas reales registradas.
"""
print("Palabras:", len(informe.split()))

Palabras: 160


## Advertencia metodológica

El valor del plan de inventario es potencial: supone que cada unidad asignada se vende hasta el límite de la demanda y no considera reposición, devoluciones ni condiciones de proveedores. La demanda de evaluación es un promedio de 22 semanas y puede ocultar la estacionalidad de fin de año. Las alertas de Isolation Forest son estadísticas: no demuestran error ni fraude.

La carta p supone una proporción de alertas estable y binomial; con 80 semanas de línea base, varias salen de los límites, por lo que una señal es un motivo para investigar y no una prueba. Los datos son de un solo mayorista y dos años: los resultados no se extrapolan a otros mercados ni al 2026. Una buena métrica no garantiza una buena decisión: las reglas se fijan antes de ver la evaluación final y el resultado se informa tal como sale.

# Rúbrica de la evaluación (resumen)

| Criterio | Ejercicios | Puntos |
|---|---|---|
| 1. Trazabilidad y calidad | 1 | 2 |
| 2. Optimización de inventario | 2 y 3 | 4 |
| 3. Anomalías y control estadístico | 4 y 5 | 4 |
| 4. Decisión integrada y lectura crítica | 6 y cierre | 3 |
| 5. Documentación y código propio | todos | 3 |
| **Total grupal** | | **16** |

Se suman 4 puntos de actividades individuales de las semanas 11 y 12. **Descuentos sobre el puntaje grupal:** puntualidad y dedicación (1), ortografía y redacción de las conclusiones (1), notebook que no ejecuta de principio a fin (2), uso de IA no declarado (2).

# Evidencia de entrega

Antes de subir a `lc6/`, confirme:

- [ ] Fuente de los datos y huella registradas; exclusiones consecutivas con sus cifras.
- [ ] Semanas completas verificadas y línea base y evaluación fijadas antes de explorar.
- [ ] Modelo de optimización entero con todas las restricciones, estado del solucionador y holguras.
- [ ] Regla intuitiva comparada con el modelo, en valor potencial y en valor realizado.
- [ ] Sensibilidad de capacidad y presupuesto (nueve combinaciones) y demanda con 13 semanas.
- [ ] Isolation Forest y umbral calculados solo con la línea base; carta p con límites por volumen.
- [ ] Cola de revisión con la capacidad declarada y sensibilidad de percentiles.
- [ ] Reglas de decisión aplicadas sin cambios; conexión entre ambas partes.
- [ ] Informe de recomendación (máximo 180 palabras) con cifras propias.
- [ ] Notebook ejecutado de inicio a fin, con URLs de GitHub y de Colab, y uso de IA declarado.

# Fuente de datos

Dua, D. y Graff, C. (s. f.). *Online Retail II* [Conjunto de datos]. UCI Machine Learning Repository. https://archive.ics.uci.edu/dataset/502/online+retail+ii

Referencias de estudio: Hillier, F. S. y Lieberman, G. J., *Introduction to Operations Research*; Liu, F. T., Ting, K. M. y Zhou, Z.-H. (2008), *Isolation Forest*; Montgomery, D. C., *Introduction to Statistical Quality Control*.